In [ ]:
import sys, json, subprocess, requests
from pathlib import Path
from google.colab import drive

def section(t):
    print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78)

drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/semeval2027/semeval2027_colab")
for p in [PROJECT / "src", PROJECT / "notebooks"]:
    if str(p) not in sys.path: sys.path.insert(0, str(p))
from semeval_task3.data import load_dataset

inventory = []

# ---------------------------------------------------------------------------
section("1. DATA WE ALREADY HAVE (in Drive)")
dev = load_dataset(PROJECT / "work/data/dev", with_gold=True)
pos = sum(dev.gold.values())
print(f"Swedish competition dev : {len(dev.usages)} usages, {len({u.word for u in dev.usages})} words, "
      f"{pos} YES ({100 * pos / len(dev.usages):.0f}%)")
print("   ROLE: final held-out judge only. Never used for adaptation training.")
inventory.append(("SemEval Swedish dev", "SV", "real task, historical", "competition terms", "FINAL EVALUATION ONLY"))

wic_dir = PROJECT / "outputs/runs/step03_english_data/20261103T165602Z_11f07da3"
if (wic_dir / "prepared_train.json").exists():
    tr = json.loads((wic_dir / "prepared_train.json").read_text()); dv = json.loads((wic_dir / "prepared_dev.json").read_text())
    print(f"\nEnglish WiC-TSV         : {len(tr)} train + {len(dv)} dev pairs (balanced YES/NO), modern English, CC BY 4.0")
    print(f"   Example: {tr[0]['marked_text'][:90]} | def: {tr[0]['definition'][:50]} | label {tr[0]['label']}")
    inventory.append(("WiC-TSV", "EN", "usage + definition, YES/NO", "CC BY 4.0", "available"))
else:
    print("\nEnglish WiC-TSV         : NOT FOUND at expected path")

dict_files = sorted((PROJECT / "outputs/runs/step06_dictionary_senses").glob("*/dictionary_senses_SV.json"))
print(f"\nSwedish dictionary senses (10 dev words): {'found' if dict_files else 'not found'}")
print("   NOTE: these cover the 10 competition words, so they must NOT be used for training.")

# ---------------------------------------------------------------------------
section("2. WIKTIONARY (kaikki.org) FOR ALL 6 TEST LANGUAGES (size check only)")
print("Content: dictionary senses with English glosses + dated quotations per sense. License: CC BY-SA.")
print("Use: can build usage + definition YES/NO pairs (as in the mock test). Bias: dictionary sense boundaries.\n")
KAIKKI = "https://kaikki.org/dictionary/{n}/kaikki.org-dictionary-{n}.jsonl"
for code, name in {"SV": "Swedish", "EN": "English", "IT": "Italian", "ES": "Spanish", "NL": "Dutch", "RU": "Russian"}.items():
    try:
        r = requests.head(KAIKKI.format(n=name), allow_redirects=True, timeout=30)
        size = int(r.headers.get("Content-Length", 0)) / 2**20
        print(f"   {name:<9} {'PASS' if r.status_code == 200 else 'FAIL'}   {size:>6.0f} MB")
    except Exception as e:
        print(f"   {name:<9} FAIL   {e}")
inventory.append(("Wiktionary quotations", "SV EN IT ES NL RU", "usage + gloss (built pairs)", "CC BY-SA", "available"))
print("\nMock-test reference (unchanged Gemma, built from this source): AUC 0.82 to 0.91 across the 6 languages.")

# ---------------------------------------------------------------------------
section("3. AXOLOTL-24 SHARED TASK DATA (inspect actual files)")
print("Closest prior task: assign each usage to one of a word's dictionary definitions.")
print("Languages: Finnish, Russian, German. Russian is one of our test languages.\n")
AX = PROJECT / "work/external/axolotl24"
if not AX.exists():
    AX.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ltgoslo/axolotl24_shared_task.git", str(AX)],
                   check=True, capture_output=True)
commit = subprocess.run(["git", "-C", str(AX), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
size_mb = sum(f.stat().st_size for f in AX.rglob("*") if f.is_file()) / 2**20
print(f"Cloned at commit {commit[:12]} ({size_mb:.1f} MB)")
lic = (AX / "LICENSE").read_text(errors="ignore").splitlines() if (AX / "LICENSE").exists() else []
print(f"Repository license (first line): {lic[0] if lic else 'NOT FOUND'}")
readme = (AX / "README.md").read_text(errors="ignore") if (AX / "README.md").exists() else ""
for kw in ["licen", "CC BY", "Creative Commons"]:
    for line in readme.splitlines():
        if kw.lower() in line.lower():
            print(f"   README mentions: {line.strip()[:150]}")
            break
print("\nData files found:")
data_files = sorted(p for p in (AX / "data").rglob("*") if p.is_file())
for f in data_files:
    print(f"   {str(f.relative_to(AX)):<55} {f.stat().st_size / 1024:>8.0f} KB")
print("\nFirst rows of each TSV (columns + 2 examples):")
for f in data_files:
    if f.suffix in (".tsv", ".csv", ".txt"):
        lines = f.read_text(errors="ignore").splitlines()
        print(f"\n--- {f.relative_to(AX)}  ({len(lines) - 1} data rows)")
        for line in lines[:3]:
            print("   ", line[:220])
inventory.append(("AXOLOTL-24", "FI RU DE", "usage -> definition assignment", "see above (repo GPL-3.0; data license to confirm)", "inspect"))

# ---------------------------------------------------------------------------
section("SUMMARY: CANDIDATE SOURCES")
print(f"{'source':<24}{'languages':<20}{'content':<34}{'license':<22}status")
for s, l, c, li, st in inventory:
    print(f"{s:<24}{l:<20}{c:<34}{li[:21]:<22}{st}")
print("\nRules for the next steps:")
print("  - The 10 Swedish competition words never enter adaptation training.")
print("  - Training and validation splits are fixed (by word) BEFORE any pairs are generated.")
print("  - Unchanged Gemma is measured on the same validation sets first.")
print("\nPaste this output. Next cell: audit each usable source in detail (labels, offsets, overlap).")

In [ ]:
import pandas as pd, re, csv
from collections import Counter

def section(t):
    print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78)

AX = PROJECT / "work/external/axolotl24"

# ---------------------------------------------------------------------------
section("1. LICENSE AND SOURCE (from the data READMEs)")
for rd in ["data/README.md", "data/finnish/README.md", "data/russian/README.md"]:
    p = AX / rd
    if p.exists():
        print(f"\n--- {rd}")
        for line in p.read_text(errors="ignore").splitlines():
            if line.strip():
                print("   ", line.strip()[:160])

# ---------------------------------------------------------------------------
section("2. LOAD AND SEPARATE ROW TYPES")
print("Rows with an example = usages. Rows without an example = sense definitions (inventory only).\n")
def load(path):
    return pd.read_csv(path, sep="\t", dtype=str, quoting=csv.QUOTE_NONE, keep_default_na=False)
files = {("FI", "train"): "data/finnish/axolotl.train.fi.tsv", ("FI", "dev"): "data/finnish/axolotl.dev.fi.tsv",
         ("FI", "test"): "data/finnish/axolotl.test.fi.gold.tsv", ("RU", "train"): "data/russian/axolotl.train.ru.tsv",
         ("RU", "dev"): "data/russian/axolotl.dev.ru.tsv", ("RU", "test"): "data/russian/axolotl.test.ru.gold.tsv"}
D = {k: load(AX / v) for k, v in files.items()}
print(f"{'lang':<5}{'split':<7}{'rows':>7}{'usages':>8}{'sense-only':>11}{'words':>7}{'with sense id':>14}"
      f"{'with offsets':>14}{'with year':>11}  periods")
for (lang, split), df in D.items():
    is_usage = df["example"].str.strip() != ""
    u = df[is_usage]
    has_sense = (u["sense_id"].str.strip() != "").sum()
    has_off = u["indices_target_token"].str.match(r"^\d+:\d+$").sum()
    has_year = u["date"].str.match(r"^\d{3,4}$").sum()
    periods = dict(Counter(u["period"]))
    print(f"{lang:<5}{split:<7}{len(df):>7}{len(u):>8}{(~is_usage).sum():>11}{df['word'].nunique():>7}"
          f"{has_sense:>14}{has_off:>14}{has_year:>11}  {periods}")

# ---------------------------------------------------------------------------
section("3. CAN WE LOCATE THE TARGET WORD WHEN OFFSETS ARE MISSING?")
print("Rule: use offsets if present; else find 'orth' in the example; else a unique word starting with the")
print("lemma's stem. Usages where the target cannot be located uniquely are dropped (counted here).\n")
def locate(row):
    ex, idx = row["example"], row["indices_target_token"]
    m = re.match(r"^(\d+):(\d+)$", idx)
    if m:
        s, e = int(m[1]), int(m[2])
        if 0 <= s < e <= len(ex): return "offsets"
    orth = row["orth"].strip()
    if orth and ex.count(orth) == 1: return "orth"
    stem = row["word"].lower()[:max(3, len(row["word"]) - 2)]
    hits = [w for w in re.finditer(r"\w+", ex) if w.group().lower().startswith(stem)]
    return "stem" if len(hits) == 1 else "not found"
for (lang, split), df in D.items():
    u = df[(df["example"].str.strip() != "") & (df["sense_id"].str.strip() != "")]
    c = Counter(u.apply(locate, axis=1))
    ok = c["offsets"] + c["orth"] + c["stem"]
    print(f"{lang} {split:<6}: {len(u):>6} labelled usages | offsets {c['offsets']:>6} | orth {c['orth']:>5} | "
          f"stem {c['stem']:>5} | NOT FOUND {c['not found']:>5} | usable {100 * ok / max(len(u), 1):5.1f}%")

# ---------------------------------------------------------------------------
section("4. CAN WE BUILD YES/NO PAIRS? (needs >= 2 senses with glosses per word)")
for (lang, split), df in D.items():
    senses = df[df["gloss"].str.strip() != ""].groupby("word")["sense_id"].nunique()
    multi = (senses >= 2).sum()
    print(f"{lang} {split:<6}: {len(senses):>5} words with glosses | {multi:>5} with >= 2 senses "
          f"({100 * multi / max(len(senses), 1):.0f}%) | median senses per word {int(senses.median()) if len(senses) else 0}")

# ---------------------------------------------------------------------------
section("5. WORD OVERLAP BETWEEN SPLITS (training must not share words with validation)")
for lang in ["FI", "RU"]:
    w = {s: set(D[(lang, s)]["word"]) for s in ["train", "dev", "test"]}
    print(f"{lang}: train-dev {len(w['train'] & w['dev'])} | train-test {len(w['train'] & w['test'])} | "
          f"dev-test {len(w['dev'] & w['test'])} shared words")
sv_words = {u.word for u in dev.usages}
print(f"Overlap with the 10 Swedish competition words: "
      f"{len(sv_words & set().union(*[set(df['word']) for df in D.values()]))} (expected 0)")

# ---------------------------------------------------------------------------
section("6. EXAMPLES (one usage + its own gloss + a sibling gloss)")
for lang in ["FI", "RU"]:
    df = D[(lang, "dev")]
    gl = df[df["gloss"].str.strip() != ""].drop_duplicates("sense_id").set_index("sense_id")["gloss"]
    u = df[(df["example"].str.strip() != "") & (df["sense_id"].isin(gl.index))]
    for _, row in u.iterrows():
        sib = df[(df["word"] == row["word"]) & (df["sense_id"] != row["sense_id"]) & (df["gloss"].str.strip() != "")]
        if len(sib):
            print(f"\n[{lang}] word: {row['word']} | date: {row['date'] or '-'} | period: {row['period']} | located by: {locate(row)}")
            print(f"   usage      : {row['example'][:160]}")
            print(f"   own sense  : {gl[row['sense_id']][:120]}   -> YES")
            print(f"   sibling    : {sib.iloc[0]['gloss'][:120]}   -> NO")
            break

section("SUMMARY")
print("Check before deciding on AXOLOTL:")
print("  1. Section 1: is the DATA license stated, and does it allow this use (with attribution)?")
print("  2. Section 3: what share of Russian usages can we locate the target in?")
print("  3. Section 4: enough words with >= 2 senses to build sibling negatives?")
print("  4. Section 5: are splits word-disjoint, and is overlap with our 10 Swedish words 0?")
print("Paste this output. Nothing has been generated, trained or modified.")

In [ ]:
import json, random
from project_utils import new_run, save_json

def section(t):
    print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78)

SEED = 2027
SPLITS = new_run(PROJECT, "step07_frozen_splits")
sv_comp_words = sorted({u.word for u in dev.usages})

section("FROZEN DESIGN (decided before any pairs are generated)")
design = {
    "train":           ["AXOLOTL FI train", "AXOLOTL RU train", "WiC-TSV EN train (minus words shared with WiC dev)"],
    "selection":       ["AXOLOTL FI dev", "AXOLOTL RU dev"],
    "validation_untouched": ["AXOLOTL FI test-gold", "AXOLOTL RU test-gold", "WiC-TSV EN dev",
                             "Wiktionary held-out sets: SV IT ES NL RU EN (validation only, never training)"],
    "final_judge":     ["SemEval Swedish dev (10 words, never trained on)"],
    "excluded_from_all_training": sv_comp_words,
    "seed": SEED,
}
for role, items in design.items():
    print(f"{role:<28}: {items}")

# ---------------------------------------------------------------------------
section("WORD LISTS PER SPLIT")
splits = {}
for lang in ["FI", "RU"]:
    for split, role in [("train", "train"), ("dev", "selection"), ("test", "validation")]:
        words = sorted(set(D[(lang, split)]["word"]))
        splits[f"AXOLOTL_{lang}_{role}"] = words
        print(f"AXOLOTL {lang} {role:<11}: {len(words):>5} words")

wic_train_words = {r["word"].casefold() for r in tr}
wic_dev_words = {r["word"].casefold() for r in dv}
shared = wic_train_words & wic_dev_words
splits["WiC_EN_train"] = sorted(wic_train_words - shared)
splits["WiC_EN_validation"] = sorted(wic_dev_words)
kept_rows = sum(r["word"].casefold() not in shared for r in tr)
print(f"WiC-TSV EN train     : {len(splits['WiC_EN_train']):>5} words ({kept_rows} of {len(tr)} pairs kept; "
      f"{len(shared)} words shared with dev removed from training)")
print(f"WiC-TSV EN validation: {len(splits['WiC_EN_validation']):>5} words ({len(dv)} pairs)")

# ---------------------------------------------------------------------------
section("LEAKAGE CHECKS")
train_sets = {"FI": set(splits["AXOLOTL_FI_train"]), "RU": set(splits["AXOLOTL_RU_train"]),
              "EN": set(splits["WiC_EN_train"])}
eval_sets = {"FI": set(splits["AXOLOTL_FI_selection"]) | set(splits["AXOLOTL_FI_validation"]),
             "RU": set(splits["AXOLOTL_RU_selection"]) | set(splits["AXOLOTL_RU_validation"]),
             "EN": set(splits["WiC_EN_validation"])}
all_ok = True
for lang in ["FI", "RU", "EN"]:
    ov = train_sets[lang] & eval_sets[lang]
    all_ok &= not ov
    print(f"{lang}: training words also in selection/validation: {len(ov):>3}  {'PASS' if not ov else 'FAIL'}")
comp_ov = set(sv_comp_words) & set().union(*train_sets.values())
all_ok &= not comp_ov
print(f"Swedish competition words in any training set: {len(comp_ov)}  {'PASS' if not comp_ov else 'FAIL'}")
print("\nNote: Wiktionary validation sets are built in the next cell and will EXCLUDE any word")
print("that appears in a training set of the same language (e.g. AXOLOTL RU train words).")

save_json(SPLITS / "splits.json", {"design": design, "word_lists": splits,
                                   "wic_train_words_removed_shared_with_dev": sorted(shared)})
section("SUMMARY")
print(f"Leakage checks : {'ALL PASS' if all_ok else 'FAIL, stop here'}")
print(f"Frozen splits  : {SPLITS / 'splits.json'}")
print("These word lists are now fixed. All later pairs, training and validation must follow them.")
print("Next cell: build the Wiktionary validation sets (SV IT ES NL RU EN), excluding training words.")

In [ ]:
import subprocess, sys, re, json, random, unicodedata
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "pymorphy3", "pymorphy3-dicts-ru"], check=True)
import pymorphy3, pandas as pd, numpy as np
from collections import Counter, defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from project_utils import new_run, save_json

def section(t):
    print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78)

RUC = new_run(PROJECT, "step07_russian_candidates")
morph = pymorphy3.MorphAnalyzer()
rng = random.Random(2027)

# ---------------------------------------------------------------------------
section("1. RUSSIAN DATA SOURCE AND LICENSE (full text, no truncation)")
readme = (AX / "data/README.md").read_text(errors="ignore")
ru_part = readme[readme.find("## Russian"): readme.find("## German")]
print(ru_part.strip())
print("\nLicense mentions anywhere in the repository READMEs:")
found = False
for p in AX.rglob("README*"):
    for line in p.read_text(errors="ignore").splitlines():
        if re.search(r"licen|creative commons|cc[- ]by", line, re.I):
            print(f"   {p.relative_to(AX)}: {line.strip()}"); found = True
print("   (none found)" if not found else "")
print("Status: no dataset-specific license stated -> document sources and cite; record as an open item.")

# ---------------------------------------------------------------------------
section("2. TARGET VERIFICATION RULES")
print("A usage is ACCEPTED only if exactly one word in the context is verified as the headword:")
print("  - verified_orth  : a whole word equals the headword or its old spelling (word boundaries enforced)")
print("  - verified_lemma : pymorphy3 gives the headword as that word's dictionary form")
print("Old spelling is normalised for this check only (ѣ->е, і->и, ѳ->ф, ѵ->и, final ъ dropped, ё->е).")
print("QUARANTINED (not used): several verified matches; only a prefix match. REJECTED: target not found,")
print("context shorter than 5 words, missing sense ID or definition, conflicting labels for the same context.\n")
OLD = str.maketrans({"ѣ": "е", "і": "и", "ѳ": "ф", "ѵ": "и", "ё": "е", "Ѣ": "Е", "І": "И", "Ѳ": "Ф", "Ѵ": "И", "Ё": "Е"})
STRESS = dict.fromkeys(map(ord, "\u0301\u0300"), None)
def norm(tok):
    t = tok.translate(STRESS).translate(OLD).lower()
    return t[:-1] if len(t) > 1 and t.endswith("ъ") else t
def lemmas(tok):
    return {p.normal_form.replace("ё", "е") for p in morph.parse(norm(tok))}

def verify(row):
    text = row["example"].translate(STRESS)
    head = norm(row["word"]); orth = norm(row["orth"]) if row["orth"].strip() else head
    toks = list(re.finditer(r"\w+", text))
    exact = [m for m in toks if norm(m.group()) in (head, orth)]
    lem = [m for m in toks if head in lemmas(m.group())]
    hits = {(m.start(), m.end()) for m in exact + lem}
    if len(hits) == 1:
        s, e = next(iter(hits)); how = "verified_orth" if exact else "verified_lemma"
        return how, text, s, e
    if len(hits) > 1:
        return "quarantine_ambiguous", text, None, None
    stem = head[:max(3, len(head) - 2)]
    if any(norm(m.group()).startswith(stem) for m in toks):
        return "quarantine_prefix_only", text, None, None
    return "reject_not_found", text, None, None

# ---------------------------------------------------------------------------
section("3. VERIFYING TRAIN AND DEV (test split is NOT touched)")
cand = {}
for split in ["train", "dev"]:
    df = D[("RU", split)]
    gloss = df[df["gloss"].str.strip() != ""].drop_duplicates("sense_id").set_index("sense_id")["gloss"].to_dict()
    sense_periods = df.groupby("sense_id")["period"].agg(lambda s: set(s)).to_dict()
    rows, reasons = [], Counter()
    for _, r in df[df["example"].str.strip() != ""].iterrows():
        if not r["sense_id"].strip():
            reasons["reject_no_sense_id"] += 1; continue
        if r["sense_id"] not in gloss:
            reasons["reject_missing_definition"] += 1; continue
        status, text, s, e = verify(r)
        if status == "verified_orth" or status == "verified_lemma":
            if len(text.split()) < 5:
                reasons["reject_fragment_under_5_words"] += 1; continue
        reasons[status] += 1
        rows.append(dict(id=r["usage_id"], word=r["word"], sense_id=r["sense_id"], period=r["period"],
                         text=text, start=s, end=e, status=status))
    # duplicate contexts: same word + same text with different senses = conflicting labels
    by_ctx = defaultdict(set)
    for x in rows:
        if x["status"].startswith("verified"): by_ctx[(x["word"], x["text"])].add(x["sense_id"])
    conflict = {k for k, v in by_ctx.items() if len(v) > 1}
    seen, final = set(), []
    for x in rows:
        k = (x["word"], x["text"])
        if x["status"].startswith("verified"):
            if k in conflict:
                x["status"] = "reject_conflicting_labels"; reasons["verified->conflicting_labels"] += 1
            elif k in seen:
                x["status"] = "reject_duplicate_context"; reasons["verified->duplicate_context"] += 1
            seen.add(k)
        final.append(x)
    cand[split] = dict(rows=final, gloss=gloss, sense_periods={k: sorted(v) for k, v in sense_periods.items()})
    acc = sum(x["status"].startswith("verified") for x in final)
    total = int((df["example"].str.strip() != "").sum())
    print(f"RU {split}: {total} usages with text -> ACCEPTED {acc} ({100 * acc / total:.1f}%)")
    for k, v in sorted(reasons.items(), key=lambda kv: -kv[1]):
        print(f"      {k:<34} {v:>5}")

# ---------------------------------------------------------------------------
section("4. PROPOSED NO-PAIRS (same-period siblings; similar definitions quarantined)")
print("For each accepted usage: a sibling sense of the same word, attested in the SAME period")
print("(old with old, new with new). Definition similarity (character n-gram cosine) >= 0.5")
print("puts the pair in quarantine for inspection instead of labelling it NO.\n")
for split in ["train", "dev"]:
    c = cand[split]; acc = [x for x in c["rows"] if x["status"].startswith("verified")]
    senses_by_word = defaultdict(set)
    for sid, per in c["sense_periods"].items():
        w = sid.rsplit("_", 1)[0]
        senses_by_word[w].add(sid)
    all_glosses = list(c["gloss"].values())
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5)).fit(all_glosses)
    neg_stats = Counter(); negs = []
    for x in acc:
        sibs = [s for s in c["gloss"] if s != x["sense_id"] and s.rsplit("_", 1)[0] == x["sense_id"].rsplit("_", 1)[0]
                and x["period"] in c["sense_periods"].get(s, [])]
        if not sibs:
            neg_stats["no same-period sibling"] += 1; continue
        sib = rng.choice(sorted(sibs))
        a, b = vec.transform([c["gloss"][x["sense_id"]], c["gloss"][sib]])
        sim = float((a @ b.T).toarray()[0, 0])
        st = "proposed_NO" if sim < 0.5 else "quarantine_similar_definitions"
        neg_stats[st] += 1
        negs.append(dict(usage=x["id"], sibling=sib, similarity=round(sim, 3), status=st))
    c["negatives"] = negs
    print(f"RU {split}: accepted usages {len(acc)} | " + " | ".join(f"{k}: {v}" for k, v in neg_stats.items()))

# ---------------------------------------------------------------------------
section("5. EXAMPLES TO INSPECT (train only)")
c = cand["train"]; acc = {x["id"]: x for x in c["rows"] if x["status"].startswith("verified")}
print("ACCEPTED, with proposed YES and NO definitions:")
for n in rng.sample([n for n in c["negatives"] if n["status"] == "proposed_NO"], 4):
    x = acc[n["usage"]]
    print(f"\n  [{x['status']}, period {x['period']}] {x['word']}")
    print(f"   context : {x['text'][:x['start']]}<t>{x['text'][x['start']:x['end']]}</t>{x['text'][x['end']:][:80]}")
    print(f"   YES def : {c['gloss'][x['sense_id']][:110]}")
    print(f"   NO def  : {c['gloss'][n['sibling']][:110]}   (similarity {n['similarity']})")
for label in ["quarantine_prefix_only", "quarantine_ambiguous"]:
    ex = [x for x in c["rows"] if x["status"] == label]
    if ex:
        x = rng.choice(ex)
        print(f"\n{label.upper()} example ({len(ex)} total): {x['word']} | {x['text'][:140]}")
q = [n for n in c["negatives"] if n["status"] == "quarantine_similar_definitions"]
if q:
    n = rng.choice(q)
    print(f"\nQUARANTINED NO-pair ({len(q)} total), similarity {n['similarity']}:")
    print(f"   true def    : {c['gloss'][acc[n['usage']]['sense_id']][:110]}")
    print(f"   sibling def : {c['gloss'][n['sibling']][:110]}")

save_json(RUC / "russian_candidates_train_dev.json", cand)
section("SUMMARY")
for split in ["train", "dev"]:
    c = cand[split]
    acc_n = sum(x["status"].startswith("verified") for x in c["rows"])
    no_n = sum(n["status"] == "proposed_NO" for n in c["negatives"])
    print(f"RU {split}: {acc_n} accepted usages (-> YES pairs) | {no_n} proposed NO pairs")
print(f"\nSaved with every row's status (accepted / quarantined / rejected): {RUC}")
print("Russian TEST split was not opened. Nothing generated for training yet; nothing trained.")
print("Paste this output. We inspect the examples together before any pair becomes training data.")

In [ ]:
# ONE CELL: fix Russian NO-pairs, save them, measure meaning similarity, print 24 pairs for review.
import sys, json, random, csv, subprocess
from pathlib import Path
from collections import Counter, defaultdict
from google.colab import drive
drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/semeval2027/semeval2027_colab")
for p in [PROJECT / "src", PROJECT / "notebooks"]:
    if str(p) not in sys.path: sys.path.insert(0, str(p))
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "sentence-transformers"], check=True)
import numpy as np, pandas as pd
from huggingface_hub import HfApi
from sentence_transformers import SentenceTransformer
from project_utils import new_run, save_json

def section(t):
    print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78)

AX = PROJECT / "work/external/axolotl24"
SRC = PROJECT / "outputs/runs/step07_russian_candidates/20261105T220554Z_8222438f/russian_candidates_train_dev.json"
assert SRC.is_file(), f"Cell 4 file not found: {SRC}"
cand = json.loads(SRC.read_text(encoding="utf-8"))
TSV = {"train": AX / "data/russian/axolotl.train.ru.tsv", "dev": AX / "data/russian/axolotl.dev.ru.tsv"}
load = lambda p: pd.read_csv(p, sep="\t", dtype=str, quoting=csv.QUOTE_NONE, keep_default_na=False)
OUT = new_run(PROJECT, "step07_russian_candidates_v2")

# ---------------------------------------------------------------------------
section("1. FIX THE NO-PAIRS")
print("Rule: NO candidate = same headword, different sense, same period, has a definition.")
print("Sense IDs shared by two headwords are a data error -> those usages are set aside.\n")
rng = random.Random(2027)
expected = {"train": 3323, "dev": 1558}
for split in ["train", "dev"]:
    c, df = cand[split], load(TSV[split])
    labelled = df[df["sense_id"].str.strip() != ""]
    words_of_sense = labelled.groupby("sense_id")["word"].agg(lambda s: sorted(set(s))).to_dict()
    shared = {sid: ws for sid, ws in words_of_sense.items() if len(ws) > 1}
    senses_of_word = defaultdict(set)
    for w, sid in labelled[["word", "sense_id"]].itertuples(index=False):
        if sid not in shared:
            senses_of_word[w].add(sid)
    for x in c["rows"]:
        if x["status"].startswith("verified") and x["sense_id"] in shared:
            x["status"] = "quarantine_sense_shared_by_headwords"
    acc = [x for x in c["rows"] if x["status"].startswith("verified")]
    negs = []
    for x in acc:
        assert x["sense_id"] in senses_of_word[x["word"]], f"{x['id']}: sense not under its headword"
        sibs = sorted(s for s in senses_of_word[x["word"]]
                      if s != x["sense_id"] and s in c["gloss"] and x["period"] in c["sense_periods"].get(s, []))
        if sibs:
            negs.append(dict(usage=x["id"], sibling=rng.choice(sibs), status="proposed_NO"))
    c["negatives"] = negs
    ok = "PASS" if len(negs) == expected[split] else f"CHECK (expected {expected[split]})"
    print(f"RU {split}: accepted usages {len(acc)} | NO-pairs {len(negs)} [{ok}] | "
          f"shared sense IDs {len(shared)}{' ' + str(list(shared.values())) if shared else ''}")
save_json(OUT / "russian_candidates_v2.json", cand)
print(f"\nSaved corrected candidates: {OUT / 'russian_candidates_v2.json'}")

# ---------------------------------------------------------------------------
section("2. MEANING SIMILARITY OF EACH NO-PAIR (multilingual-e5-large, CPU)")
print("Higher = the two definitions mean more similar things (risk that NO is actually also correct).")
print("The model download (2.2 GB) can sit at 0% for up to a minute before it starts.\n")
MODEL_ID = "intfloat/multilingual-e5-large"
rev = HfApi().model_info(MODEL_ID).sha
enc = SentenceTransformer(MODEL_ID, revision=rev, device="cpu")
print(f"Model loaded: PASS (revision {rev[:12]})")
for split in ["train", "dev"]:
    c = cand[split]
    acc = {x["id"]: x for x in c["rows"] if x["status"].startswith("verified")}
    gl = sorted({c["gloss"][acc[n["usage"]]["sense_id"]] for n in c["negatives"]} | {c["gloss"][n["sibling"]] for n in c["negatives"]})
    vec = dict(zip(gl, enc.encode(["query: " + g for g in gl], batch_size=32, normalize_embeddings=True, show_progress_bar=True)))
    for n in c["negatives"]:
        n["meaning_sim"] = float(np.clip(vec[c["gloss"][acc[n["usage"]]["sense_id"]]] @ vec[c["gloss"][n["sibling"]]], -1, 1))
    print(f"RU {split}: similarity percentiles 10/25/50/75/90/95: "
          f"{np.percentile([n['meaning_sim'] for n in c['negatives']], [10, 25, 50, 75, 90, 95]).round(3)}")
save_json(OUT / "russian_candidates_v2_with_similarity.json", cand)

# ---------------------------------------------------------------------------
section("3. 24 TRAIN PAIRS FOR REVIEW (4 random pairs from each of 6 similarity bands)")
print("For each: could the NO definition ALSO describe the marked word? If yes, that NO label is wrong.\n")
rng = random.Random(2027)
c = cand["train"]
acc = {x["id"]: x for x in c["rows"] if x["status"].startswith("verified")}
pairs = sorted(c["negatives"], key=lambda n: (n["meaning_sim"], n["usage"]))
review = []
for band, idx in enumerate(np.array_split(np.arange(len(pairs)), 6), 1):
    lo, hi = pairs[int(idx[0])]["meaning_sim"], pairs[int(idx[-1])]["meaning_sim"]
    for i in rng.sample([int(i) for i in idx], 4):
        n, x = pairs[i], acc[pairs[i]["usage"]]
        s, e = x["start"], x["end"]
        review.append(dict(n=len(review) + 1, band=band, usage=x["id"], word=x["word"], sim=n["meaning_sim"]))
        print(f"#{len(review):<2} band {band} ({lo:.3f}-{hi:.3f}) | sim {n['meaning_sim']:.3f} | {x['word']} ({x['period']})")
        print(f"     context : ...{x['text'][max(0, s - 80):s]}<t>{x['text'][s:e]}</t>{x['text'][e:e + 80]}...")
        print(f"     YES def : {c['gloss'][x['sense_id']][:160]}")
        print(f"     NO def  : {c['gloss'][n['sibling']][:160]}\n")
save_json(OUT / "review_24.json", review)

section("SUMMARY")
print(f"Everything saved in: {OUT}")
print("No labels changed, nothing trained, Russian test split not used.")
print("Paste this whole output.")

In [ ]:
import sys, json, random, hashlib
from pathlib import Path
from collections import Counter
from google.colab import drive
drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/semeval2027/semeval2027_colab")
for p in [PROJECT / "src", PROJECT / "notebooks"]:
    if str(p) not in sys.path: sys.path.insert(0, str(p))
from project_utils import new_run, save_json

def section(t):
    print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78)

CAND = PROJECT / "outputs/runs/step07_russian_candidates_v2/20261105T222545Z_1492689c/russian_candidates_v2_with_similarity.json"
SPLITS = PROJECT / "outputs/runs/step07_frozen_splits/20261105T220410Z_0c2b5b52/splits.json"
WIC = PROJECT / "outputs/runs/step03_english_data/20261103T165602Z_11f07da3/prepared_train.json"
for p in (CAND, SPLITS, WIC):
    assert p.is_file(), f"Missing input: {p}"
cand = json.loads(CAND.read_text(encoding="utf-8"))
splits = json.loads(SPLITS.read_text(encoding="utf-8"))
WL = splits["word_lists"]
wic_train = json.loads(WIC.read_text(encoding="utf-8"))
SIM_MAX, SEED = 0.93, 2027
OUT = new_run(PROJECT, "step07_pilot_dataset")
LANG = {"RU": "Russian", "EN": "English"}

# Final prompt, word for word (same template as the frozen system).
def messages(lang, word, definition, marked_text, when):
    L = LANG[lang]
    return [{"role": "system", "content": f"You are an expert lexicographer of historical {L}."},
            {"role": "user", "content": (
                f"Target word (lemma): {word}\n"
                f"Sense definition ({L}): {definition}\n"
                f"Sentence from {when}: {marked_text}\n\n"
                "Does the word marked with <t> and </t> in the sentence express "
                "the sense given by the definition? Answer only Yes or No.")}]

# ---------------------------------------------------------------------------
section("1. RUSSIAN PAIRS (modern period only; near-synonym NO-pairs removed)")
print(f"Filters fixed before building: period == 'new'; meaning similarity < {SIM_MAX}.")
print("Each kept usage -> 2 examples: own definition (YES) and same-period sibling (NO).\n")
def build_ru(split):
    c = cand[split]
    acc = {x["id"]: x for x in c["rows"] if x["status"].startswith("verified")}
    rows, stats = [], Counter()
    for n in c["negatives"]:
        x = acc[n["usage"]]
        if x["period"] != "new":
            stats["dropped: old period"] += 1; continue
        if n["meaning_sim"] >= SIM_MAX:
            stats["dropped: near-synonym definitions"] += 1; continue
        s, e = x["start"], x["end"]
        assert isinstance(s, int) and isinstance(e, int) and 0 <= s < e <= len(x["text"]), x["id"]
        marked = x["text"][:s] + "<t>" + x["text"][s:e] + "</t>" + x["text"][e:]
        for label, d in [(1, c["gloss"][x["sense_id"]]), (0, c["gloss"][n["sibling"]])]:
            rows.append(dict(id=f"{x['id']}|{'yes' if label else 'no'}", source="AXOLOTL-24", lang="RU",
                             split=split, word=x["word"], label=label,
                             messages=messages("RU", x["word"], d, marked, "modern times")))
        stats["kept usages"] += 1
    return rows, stats
ru_train, st_tr = build_ru("train")
ru_sel, st_sel = build_ru("dev")
for name, st, rows in [("train", st_tr, ru_train), ("dev -> selection", st_sel, ru_sel)]:
    print(f"RU {name:<17}: " + " | ".join(f"{k} {v}" for k, v in st.items()) + f"  ->  {len(rows)} examples")

# ---------------------------------------------------------------------------
section("2. ENGLISH PAIRS (WiC-TSV train, words shared with WiC dev already excluded)")
en_ok = set(WL["WiC_EN_train"])
en_train = [dict(id=r["id"], source="WiC-TSV", lang="EN", split="train", word=r["word"], label=int(r["label"]),
                 messages=messages("EN", r["word"], r["definition"], r["marked_text"], "modern times"))
            for r in wic_train if r["word"].casefold() in en_ok]
print(f"EN train: {len(en_train)} examples (of {len(wic_train)}; pairs with dev-shared words removed)")

# ---------------------------------------------------------------------------
section("3. LEAKAGE AND SANITY CHECKS")
checks = {}
ru_tr_words = {r["word"] for r in ru_train}; ru_sel_words = {r["word"] for r in ru_sel}
checks["RU train words all in AXOLOTL train list"] = ru_tr_words <= set(WL["AXOLOTL_RU_train"])
checks["RU selection words all in AXOLOTL dev list"] = ru_sel_words <= set(WL["AXOLOTL_RU_selection"])
checks["RU train vs selection: no shared words"] = not (ru_tr_words & ru_sel_words)
checks["RU train vs Russian TEST words: none shared"] = not (ru_tr_words & set(WL["AXOLOTL_RU_validation"]))
checks["EN train vs WiC dev words: none shared"] = not ({r["word"].casefold() for r in en_train} & set(WL["WiC_EN_validation"]))
sv = set(splits["design"]["excluded_from_all_training"])
checks["10 Swedish competition words: not in training"] = not (sv & (ru_tr_words | {r["word"] for r in en_train}))
all_ids = [r["id"] for r in ru_train + ru_sel + en_train]
checks["all example IDs unique"] = len(all_ids) == len(set(all_ids))
def sentence_part(r):
    content = r["messages"][1]["content"]
    return content.split("Sentence from ", 1)[1].rsplit("\n\nDoes the word marked", 1)[0]
checks["every example has exactly one <t> target"] = all(
    sentence_part(r).count("<t>") == 1 and sentence_part(r).count("</t>") == 1 for r in ru_train + ru_sel + en_train)
for k, v in checks.items():
    print(f"  {k:<48} {'PASS' if v else 'FAIL'}")
assert all(checks.values()), "A check failed: stop here and paste the output."

# ---------------------------------------------------------------------------
section("4. FROZEN PILOT DATASET")
train = ru_train + en_train
random.Random(SEED).shuffle(train)
def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    return hashlib.sha256(path.read_bytes()).hexdigest()
h_train = write_jsonl(OUT / "pilot_train.jsonl", train)
h_sel = write_jsonl(OUT / "pilot_selection_ru_dev.jsonl", ru_sel)
for name, rows in [("TRAIN", train), ("SELECTION (RU dev)", ru_sel)]:
    lab = Counter(r["label"] for r in rows); lang = Counter(r["lang"] for r in rows)
    chars = [len(r["messages"][1]["content"]) for r in rows]
    print(f"{name:<19}: {len(rows):>5} examples | YES {lab[1]} / NO {lab[0]} | languages {dict(lang)} | "
          f"prompt length median {int(sorted(chars)[len(chars)//2])} chars, max {max(chars)}")

print("\nExample training records (exactly what Gemma will see):")
for lang in ["RU", "EN"]:
    r = next(r for r in train if r["lang"] == lang)
    print(f"\n--- {lang} | label {'YES' if r['label'] else 'NO'} | {r['id']}")
    print(r["messages"][1]["content"])

save_json(OUT / "manifest.json", {
    "inputs": {"russian_candidates": str(CAND), "splits": str(SPLITS), "wic_train": str(WIC)},
    "filters": {"russian_period": "new only", "meaning_similarity_max": SIM_MAX,
                "reason": "README warns XIX-century examples are noisy; review found dictionary metatext; near-synonym NO-pairs"},
    "review_of_24_no_pairs": {"correct": 19, "uncertain": 2, "wrong_no": 2, "garbage_context": 1,
                              "wrong": ["ворох (0.853)", "дошлый (0.953)"], "garbage": ["согласие (old, Dahl metatext)"]},
    "prompt": "final system prompt; Russian/English definitions labelled by language; 'Sentence from modern times'",
    "counts": {"train": len(train), "selection": len(ru_sel)},
    "sha256": {"pilot_train.jsonl": h_train, "pilot_selection_ru_dev.jsonl": h_sel},
    "checks": checks, "seed": SEED, "russian_test_split_used": False})
section("SUMMARY")
print(f"Frozen files in {OUT}:")
print(f"   pilot_train.jsonl              {len(train)} examples   sha256 {h_train[:12]}...")
print(f"   pilot_selection_ru_dev.jsonl   {len(ru_sel)} examples   sha256 {h_sel[:12]}...")
print("Next: build the untouched VALIDATION sets (RU test-gold, EN WiC dev, Wiktionary SV/IT/ES/NL),")
print("then score UNCHANGED Gemma on them before any training.")

In [ ]:
# CELL 7 — Freeze English/Russian pilot evaluation
# No Wiktionary data. No Gemma inference. No changes to training files.

import sys, json, csv, re, random, hashlib, subprocess
from pathlib import Path
from collections import defaultdict, Counter
from google.colab import drive

drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/semeval2027/semeval2027_colab")
for folder in (PROJECT / "src", PROJECT / "notebooks"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "pymorphy3", "pymorphy3-dicts-ru", "sentence-transformers"],
    check=True,
)

import pandas as pd
import numpy as np
import torch
import pymorphy3
from huggingface_hub import HfApi
from sentence_transformers import SentenceTransformer
from project_utils import new_run, save_json

SPLIT_FILE = (
    PROJECT / "outputs/runs/step07_frozen_splits"
    / "20261105T220410Z_0c2b5b52/splits.json"
)
EN_FILE = (
    PROJECT / "outputs/runs/step03_english_data"
    / "20261103T165602Z_11f07da3/prepared_dev.json"
)
RU_FILE = (
    PROJECT / "work/external/axolotl24"
    / "data/russian/axolotl.test.ru.gold.tsv"
)

for path in (SPLIT_FILE, EN_FILE, RU_FILE):
    assert path.is_file(), f"Missing file: {path}"

splits = json.loads(SPLIT_FILE.read_text(encoding="utf-8"))
WL = splits["word_lists"]
print("Saved word-list names:", sorted(WL))

def fold(word):
    return word.strip().casefold().replace("ё", "е")

required = {
    "WiC_EN_train", "AXOLOTL_RU_train", "AXOLOTL_RU_validation"
}
assert required <= set(WL), (
    f"Missing expected word lists: {sorted(required - set(WL))}. "
    "Stop and paste the word-list names above."
)

# Conservatively exclude EVERY other Russian word list.
# This includes training and checkpoint-selection words.
ru_excluded_lists = [
    key for key in WL
    if key.startswith("AXOLOTL_RU_")
    and key != "AXOLOTL_RU_validation"
]
assert any(key != "AXOLOTL_RU_train" for key in ru_excluded_lists), (
    "No separate Russian selection word list found. "
    "Stop and paste the word-list names above."
)

ru_excluded = {
    fold(w) for key in ru_excluded_lists for w in WL[key]
}
ru_allowed = {fold(w) for w in WL["AXOLOTL_RU_validation"]}
en_train = {fold(w) for w in WL["WiC_EN_train"]}

assert not (ru_allowed & ru_excluded), (
    "Russian evaluation words overlap training/selection words."
)
print("Russian excluded lists:", ru_excluded_lists)

def messages(language, word, definition, marked_text, when):
    return [
        {
            "role": "system",
            "content": f"You are an expert lexicographer of historical {language}.",
        },
        {
            "role": "user",
            "content": (
                f"Target word (lemma): {word}\n"
                f"Sense definition ({language}): {definition}\n"
                f"Sentence from {when}: {marked_text}\n\n"
                "Does the word marked with <t> and </t> in the sentence "
                "express the sense given by the definition? "
                "Answer only Yes or No."
            ),
        },
    ]

# 1. English: preserve the existing labeled development examples.
english = json.loads(EN_FILE.read_text(encoding="utf-8"))
assert english, "English development file is empty."
overlap = {fold(r["word"]) for r in english} & en_train
assert not overlap, (
    f"English dev shares {len(overlap)} training words. "
    "Stop: do not silently filter or change the frozen training set."
)

en_rows = []
for r in english:
    label = int(r["label"])
    assert label in (0, 1)
    assert r["marked_text"].count("<t>") == 1
    assert r["marked_text"].count("</t>") == 1
    en_rows.append({
        "id": str(r["id"]),
        "set": "EN_wic_dev",
        "lang": "EN",
        "word": r["word"],
        "label": label,
        "definition": r["definition"],
        "marked_text": r["marked_text"],
        "label_source": "WiC-TSV supplied label",
        "messages": messages(
            "English", r["word"], r["definition"],
            r["marked_text"], "modern times",
        ),
    })

# 2. Russian: modern subset, conservative matching and deduplication.
df = pd.read_csv(
    RU_FILE, sep="\t", dtype=str,
    quoting=csv.QUOTE_NONE, keep_default_na=False,
)
assert {fold(w) for w in df["word"]} <= ru_allowed, (
    "Russian file contains words outside the frozen evaluation list."
)

morph = pymorphy3.MorphAnalyzer()
stress = dict.fromkeys(map(ord, "\u0301\u0300"), None)
old_letters = str.maketrans({
    "ѣ": "е", "і": "и", "ѳ": "ф", "ѵ": "и", "ё": "е",
})

def norm(text):
    text = text.translate(stress).lower().translate(old_letters)
    return text[:-1] if len(text) > 1 and text.endswith("ъ") else text

from functools import lru_cache

@lru_cache(maxsize=50000)
def lemma_options(token):
    return {
        norm(p.normal_form)
        for p in morph.parse(norm(token))
    }

def locate(word, orth, raw_text):
    # Offsets explicitly refer to this processed text.
    text = raw_text.translate(stress)
    head = norm(word)
    alternatives = {head, norm(orth)} - {""}
    hits = {}

    for match in re.finditer(r"\w+", text):
        span = (match.start(), match.end())
        if norm(match.group()) in alternatives:
            hits[span] = "exact_normalized"
        elif head in lemma_options(match.group()):
            hits[span] = "morphology_candidate"

    if len(hits) != 1:
        return None
    (start, end), method = next(iter(hits.items()))
    return text, start, end, method

# Use explicit word/sense mappings, never sense-ID prefixes.
owners = defaultdict(set)
definitions = defaultdict(set)
periods = defaultdict(set)
for r in df.to_dict("records"):
    if not r["sense_id"].strip():
        continue
    key = (r["word"], r["sense_id"])
    owners[r["sense_id"]].add(r["word"])
    periods[key].add(r["period"])
    if r["gloss"].strip():
        definitions[key].add(r["gloss"].strip())

gloss = {
    key: next(iter(values))
    for key, values in definitions.items()
    if len(values) == 1 and len(owners[key[1]]) == 1
}
by_word = defaultdict(list)
for word, sid in gloss:
    by_word[word].append(sid)

stats = Counter()
accepted = []
for r in df.to_dict("records"):
    if not r["example"].strip():
        stats["no_context"] += 1
        continue
    if r["period"] != "new":
        stats["excluded_old_period"] += 1
        continue
    key = (r["word"], r["sense_id"])
    if key not in gloss:
        stats["missing_ambiguous_or_shared_definition"] += 1
        continue

    match = locate(r["word"], r["orth"], r["example"])
    if match is None:
        stats["target_not_uniquely_located"] += 1
        continue
    text, start, end, method = match
    if len(text.split()) < 5:
        stats["short_context"] += 1
        continue

    accepted.append({
        "source_id": r["usage_id"],
        "word": r["word"],
        "sense_id": r["sense_id"],
        "raw_text": r["example"],
        "text": text,
        "start": start,
        "end": end,
        "target_method": method,
    })

context_senses = defaultdict(set)
for x in accepted:
    context_senses[(x["word"], x["text"])].add(x["sense_id"])

rng = random.Random(2027)
seen = set()
proposals = []
for x in accepted:
    context_key = (x["word"], x["text"])
    if len(context_senses[context_key]) > 1:
        stats["conflicting_context"] += 1
        continue
    if context_key in seen:
        stats["duplicate_context"] += 1
        continue
    seen.add(context_key)

    siblings = sorted(
        sid for sid in by_word[x["word"]]
        if sid != x["sense_id"]
        and "new" in periods[(x["word"], sid)]
    )
    if not siblings:
        stats["no_same_period_sibling"] += 1
        continue

    sibling = rng.choice(siblings)
    proposals.append((
        x,
        sibling,
        gloss[(x["word"], x["sense_id"])],
        gloss[(x["word"], sibling)],
    ))

assert proposals, "No Russian pairs survived preparation."

# Preserve the proposed similarity filter from your earlier Cell 7.
# This is a heuristic filter, NOT proof that remaining NO labels are correct.
MODEL_ID = "intfloat/multilingual-e5-large"
REVISION = HfApi().model_info(MODEL_ID).sha
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
encoder = SentenceTransformer(
    MODEL_ID, revision=REVISION, device=DEVICE
)
texts = sorted({g for _, _, a, b in proposals for g in (a, b)})
vectors = dict(zip(
    texts,
    encoder.encode(
        ["query: " + g for g in texts],
        batch_size=16,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True,
    ),
))

ru_rows = []
for x, sibling, own, other in proposals:
    similarity = float(np.dot(vectors[own], vectors[other]))
    assert np.isfinite(similarity)
    if similarity >= 0.93:
        stats["high_definition_similarity"] += 1
        continue

    start, end = x["start"], x["end"]
    marked = (
        x["text"][:start] + "<t>" + x["text"][start:end]
        + "</t>" + x["text"][end:]
    )
    for label, definition, sid in (
        (1, own, x["sense_id"]),
        (0, other, sibling),
    ):
        ru_rows.append({
            **x,
            "id": f"{x['source_id']}|{'yes' if label else 'no'}",
            "set": "RU_axolotl_test_derived",
            "lang": "RU",
            "period": "new",
            "label": label,
            "definition": definition,
            "candidate_sense_id": sid,
            "marked_text": marked,
            "definition_similarity": similarity,
            "label_source": (
                "source-assigned sense" if label
                else "constructed same-word sibling negative; not human-verified"
            ),
            "messages": messages(
                "Russian", x["word"], definition, marked, "modern times"
            ),
        })

# 3. Validate and save. No examples or model scores are printed.
sets = {
    "EN_wic_dev": en_rows,
    "RU_axolotl_test_derived": ru_rows,
}
for name, rows in sets.items():
    assert rows, f"{name} is empty"
    assert len({r["id"] for r in rows}) == len(rows), (
        f"{name}: duplicate IDs"
    )
    assert {r["label"] for r in rows} == {0, 1}

assert not ({fold(r["word"]) for r in ru_rows} & ru_excluded)
assert not ({fold(r["word"]) for r in en_rows} & en_train)

OUT = new_run(PROJECT, "step07_pilot_evaluation")
hashes = {}
for name, rows in sets.items():
    path = OUT / f"{name}.jsonl"
    with path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")
    hashes[path.name] = hashlib.sha256(path.read_bytes()).hexdigest()

save_json(OUT / "manifest.json", {
    "seed": 2027,
    "source_sha256": {
        str(p): hashlib.sha256(p.read_bytes()).hexdigest()
        for p in (SPLIT_FILE, EN_FILE, RU_FILE)
    },
    "output_sha256": hashes,
    "counts": {name: len(rows) for name, rows in sets.items()},
    "russian_excluded_word_lists": ru_excluded_lists,
    "russian_filter_counts": dict(stats),
    "embedding_model": MODEL_ID,
    "embedding_revision": REVISION,
    "similarity_exclusion_threshold": 0.93,
    "limitations": [
        "Russian negatives are constructed, not individually human-verified.",
        "Morphology-based target matches can be ambiguous linguistically.",
        "Russian evaluation includes only the new period.",
        "English development data has been evaluated in earlier experiments.",
        "Word separation checked; exhaustive cross-source text overlap not checked.",
        "These sets do not establish performance in all six languages.",
    ],
    "checkpoint_selection": "Use the separate Russian selection set only.",
    "evaluation_policy": (
        "Do not tune prompts, thresholds, or checkpoints on these evaluation labels. "
        "Report F1, precision, recall, AUC, and paired uncertainty."
    ),
})

print("\nWORD-SEPARATION AND FILE CHECKS: PASS")
for name, rows in sets.items():
    counts = Counter(r["label"] for r in rows)
    print(
        f"{name}: {len(rows)} pairs; "
        f"YES={counts[1]}, NO={counts[0]}; "
        f"words={len({r['word'] for r in rows})}"
    )
print("Russian exclusion counts:", dict(stats))
print("\nSAVED:", OUT)
print("No training files changed. No Gemma run performed.")

In [ ]:
import torch

assert torch.cuda.is_available(), "Select a GPU runtime first."

gpu = torch.cuda.get_device_properties(0)
free, total = torch.cuda.mem_get_info()

print("GPU:", gpu.name)
print(f"VRAM: {total / 2**30:.1f} GiB total")
print(f"Free VRAM: {free / 2**30:.1f} GiB")
print("PyTorch:", torch.__version__)
print("BF16 supported:", torch.cuda.is_bf16_supported())

assert total / 2**30 >= 75, (
    "For the original BF16 Gemma baseline, use A100 80 GB or G4 96 GB."
)
assert torch.cuda.is_bf16_supported(), "BF16 support required."

print("\nREADY for unchanged Gemma baseline loading.")

In [ ]:
import sys, subprocess, json, time, hashlib, shutil
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

# Match the Transformers version used for your original Gemma run.
# This does not request a PyTorch upgrade.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "transformers==5.22.1", "accelerate", "safetensors"],
    check=True,
)

import torch
import transformers
from transformers import AutoProcessor, AutoModelForMultimodalLM

assert transformers.__version__ == "5.22.1", (
    "An older Transformers version is already imported. "
    "Restart the session, then rerun this cell."
)
assert torch.cuda.is_available()
assert "gemma_model" not in globals(), (
    "gemma_model already exists. Do not load a second copy."
)

PROJECT = Path("/content/drive/MyDrive/semeval2027/semeval2027_colab")
for folder in (PROJECT / "src", PROJECT / "notebooks"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))

from project_utils import new_run, save_json

EVAL_DIR = (
    PROJECT / "outputs/runs/step07_pilot_evaluation"
    / "20261105T223700Z_5259f45a"
)
manifest = json.loads((EVAL_DIR / "manifest.json").read_text())

# Verify that the saved evaluation inputs have not changed.
for filename, expected_hash in manifest["output_sha256"].items():
    path = EVAL_DIR / filename
    assert path.is_file(), f"Missing evaluation file: {path}"
    actual_hash = hashlib.sha256(path.read_bytes()).hexdigest()
    assert actual_hash == expected_hash, f"File changed: {filename}"

MODEL_ID = "google/gemma-4-31B-it"
MODEL_REVISION = "842da3794eaa0b77d5f08bae87a17459d91ff475"
CACHE = Path("/content/hf_cache")
CACHE.mkdir(parents=True, exist_ok=True)

BASELINE_RUN = new_run(PROJECT, "step08_gemma_unchanged_baseline")

free_gpu, total_gpu = torch.cuda.mem_get_info()
print("Evaluation file hashes: PASS")
print("GPU:", torch.cuda.get_device_name(0))
print(f"Free GPU memory: {free_gpu / 2**30:.1f} GiB")
print(f"Free temporary disk: {shutil.disk_usage('/content').free / 2**30:.1f} GiB")
print("Model cache:", CACHE)
print("Run directory:", BASELINE_RUN)

assert free_gpu / 2**30 >= 65, (
    "Insufficient free GPU memory for this BF16 baseline. "
    "Release other loaded models before continuing."
)

started = time.perf_counter()

gemma_processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    cache_dir=str(CACHE),
)
gemma_tokenizer = gemma_processor.tokenizer
gemma_tokenizer.padding_side = "left"
if gemma_tokenizer.pad_token_id is None:
    gemma_tokenizer.pad_token = gemma_tokenizer.eos_token

gemma_model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    cache_dir=str(CACHE),
    dtype=torch.bfloat16,
    device_map={"": 0},
    attn_implementation="sdpa",
)
gemma_model.eval()
gemma_model.config.use_cache = False

assert all(p.device.type == "cuda" for p in gemma_model.parameters())
assert not getattr(gemma_model, "peft_config", None), (
    "Unexpected adapter on the unchanged baseline."
)

YES_IDS = gemma_tokenizer.encode("Yes", add_special_tokens=False)
NO_IDS = gemma_tokenizer.encode("No", add_special_tokens=False)

assert YES_IDS == [10784] and NO_IDS == [3771], (
    f"Token IDs differ from the original run: Yes={YES_IDS}, No={NO_IDS}"
)
YES_ID, NO_ID = YES_IDS[0], NO_IDS[0]

print("\nMODEL LOADED")
print("Class:", type(gemma_model).__name__)
print(f"Download + load: {time.perf_counter() - started:.1f}s")
print(f"GPU allocated: {torch.cuda.memory_allocated() / 2**30:.1f} GiB")

# Smoke test: one example per language, no labels used.
smoke_results = []
for dataset_name in ("EN_wic_dev", "RU_axolotl_test_derived"):
    with (EVAL_DIR / f"{dataset_name}.jsonl").open(encoding="utf-8") as handle:
        row = json.loads(next(handle))

    inputs = gemma_processor.apply_chat_template(
        row["messages"],
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True,
        enable_thinking=False,
        processor_kwargs={"padding": True, "truncation": False},
    ).to("cuda")

    with torch.inference_mode():
        output = gemma_model(
            **inputs,
            use_cache=False,
            logits_to_keep=1,
        )
        logits = output.logits[0, -1].float()
        assert torch.isfinite(logits).all()
        margin = float((logits[YES_ID] - logits[NO_ID]).item())
        probabilities = torch.softmax(logits, dim=-1)
        yes_no_mass = float(
            (probabilities[YES_ID] + probabilities[NO_ID]).item()
        )

    result = {
        "dataset": dataset_name,
        "id": row["id"],
        "tokens": int(inputs["input_ids"].shape[-1]),
        "yes_minus_no": margin,
        "yes_no_probability_mass": yes_no_mass,
    }
    smoke_results.append(result)
    print(
        f"{dataset_name}: tokens={result['tokens']}, "
        f"Yes−No={margin:.4f}, Yes/No mass={yes_no_mass:.4f}"
    )
    del inputs, output, logits, probabilities

save_json(BASELINE_RUN / "setup.json", {
    "model": MODEL_ID,
    "revision": MODEL_REVISION,
    "dtype": "bfloat16",
    "quantized": False,
    "adapter": None,
    "attention": "sdpa",
    "thinking": False,
    "score": "logit(Yes) minus logit(No)",
    "yes_token_id": YES_ID,
    "no_token_id": NO_ID,
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "evaluation_directory": str(EVAL_DIR),
    "evaluation_sha256": manifest["output_sha256"],
})
save_json(BASELINE_RUN / "smoke_test.json", smoke_results)

print("\nSAVED:", BASELINE_RUN)
print("Next: score all 3,481 evaluation examples with resumable saving.")
print("No training performed. No F1 calculated yet.")

In [ ]:
import json, time, hashlib
import numpy as np
import torch
from tqdm.auto import tqdm
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score
)
from project_utils import save_json

assert "gemma_model" in globals(), "Run the model-loading cell first."
assert "BASELINE_RUN" in globals() and "EVAL_DIR" in globals()

gemma_model.eval()
gemma_tokenizer.padding_side = "left"

BATCH_SIZE = 2
THRESHOLD = 10.5  # Existing Swedish-derived threshold; NOT tuned here.
DATASETS = ("EN_wic_dev", "RU_axolotl_test_derived")
CHECKPOINT = BASELINE_RUN / "scores_checkpoint.json"

# Verify evaluation files against the model-loading record.
setup = json.loads((BASELINE_RUN / "setup.json").read_text())
identity = hashlib.sha256(
    json.dumps(setup, sort_keys=True).encode("utf-8")
).hexdigest()

datasets = {}
for name in DATASETS:
    path = EVAL_DIR / f"{name}.jsonl"
    expected = setup["evaluation_sha256"][path.name]
    assert hashlib.sha256(path.read_bytes()).hexdigest() == expected, (
        f"Evaluation file changed: {path.name}"
    )
    with path.open(encoding="utf-8") as handle:
        rows = [json.loads(line) for line in handle if line.strip()]
    assert len({str(r["id"]) for r in rows}) == len(rows)
    datasets[name] = rows

# Load an existing checkpoint only if it belongs to this exact setup.
if CHECKPOINT.exists():
    state = json.loads(CHECKPOINT.read_text())
    assert state["setup_sha256"] == identity, (
        "Checkpoint belongs to a different model/evaluation setup."
    )
else:
    state = {
        "setup_sha256": identity,
        "scores": {name: {} for name in DATASETS},
        "inference_seconds": 0.0,
    }

for name, rows in datasets.items():
    valid_ids = {str(r["id"]) for r in rows}
    assert set(state["scores"][name]) <= valid_ids
    for saved in state["scores"][name].values():
        assert np.isfinite(saved["yes_minus_no"])
        assert np.isfinite(saved["yes_no_mass"])

total = sum(len(rows) for rows in datasets.values())
already = sum(len(state["scores"][name]) for name in DATASETS)
print(f"Total: {total}; already saved: {already}; remaining: {total - already}")

started = time.perf_counter()

try:
    for name, rows in datasets.items():
        pending = [
            r for r in rows
            if str(r["id"]) not in state["scores"][name]
        ]

        with tqdm(total=len(pending), desc=name, unit="example") as bar:
            for offset in range(0, len(pending), BATCH_SIZE):
                batch = pending[offset:offset + BATCH_SIZE]

                # Only messages go into the model. Labels are not used.
                inputs = gemma_processor.apply_chat_template(
                    [r["messages"] for r in batch],
                    tokenize=True,
                    return_dict=True,
                    return_tensors="pt",
                    add_generation_prompt=True,
                    enable_thinking=False,
                    processor_kwargs={
                        "padding": True,
                        "truncation": False,
                    },
                ).to("cuda")

                lengths = inputs["attention_mask"].sum(dim=1).tolist()

                with torch.inference_mode():
                    output = gemma_model(
                        **inputs,
                        use_cache=False,
                        logits_to_keep=1,
                    )
                    # Left padding means the final position is real input.
                    logits = output.logits[:, -1, :].float()
                    assert torch.isfinite(logits).all(), "Non-finite logits"

                    margins = (
                        logits[:, YES_ID] - logits[:, NO_ID]
                    ).cpu().tolist()

                    log_probs = torch.log_softmax(logits, dim=-1)
                    masses = (
                        log_probs[:, YES_ID].exp()
                        + log_probs[:, NO_ID].exp()
                    ).cpu().tolist()

                for row, margin, mass, length in zip(
                    batch, margins, masses, lengths
                ):
                    state["scores"][name][str(row["id"])] = {
                        "yes_minus_no": float(margin),
                        "yes_no_mass": float(mass),
                        "input_tokens": int(length),
                    }

                del inputs, output, logits, log_probs
                bar.update(len(batch))

                # Save every 20 examples.
                if (offset // BATCH_SIZE + 1) % 10 == 0:
                    save_json(CHECKPOINT, state)

        save_json(CHECKPOINT, state)

finally:
    # Also saves completed batches on a normal interrupt or Python error.
    state["inference_seconds"] += time.perf_counter() - started
    save_json(CHECKPOINT, state)

# Require complete coverage before calculating any final metrics.
for name, rows in datasets.items():
    assert set(state["scores"][name]) == {str(r["id"]) for r in rows}, (
        f"{name}: incomplete coverage. Rerun this cell to resume."
    )

results = {
    "threshold": THRESHOLD,
    "threshold_source": "Previously frozen Swedish Gemma-v1 threshold",
    "threshold_tuned_on_these_sets": False,
    "metrics_scope": "Pooled binary metrics within each external dataset",
    "datasets": {},
}

print(f"\nBASELINE RESULTS — fixed threshold {THRESHOLD}")
print("These are external pooled F1 scores, not official Swedish per-word F1.\n")

for name, rows in datasets.items():
    y = np.array([int(r["label"]) for r in rows])
    margins = np.array([
        state["scores"][name][str(r["id"])]["yes_minus_no"]
        for r in rows
    ])
    masses = np.array([
        state["scores"][name][str(r["id"])]["yes_no_mass"]
        for r in rows
    ])
    predictions = (margins >= THRESHOLD).astype(int)

    report = {
        "examples": len(rows),
        "words": len({r["word"] for r in rows}),
        "positives": int(y.sum()),
        "predicted_positives": int(predictions.sum()),
        "precision": float(precision_score(y, predictions, zero_division=0)),
        "recall": float(recall_score(y, predictions, zero_division=0)),
        "f1": float(f1_score(y, predictions, zero_division=0)),
        "auc": float(roc_auc_score(y, margins)),
        "yes_no_mass_below_0_90": int((masses < 0.90).sum()),
    }
    results["datasets"][name] = report

    # Preserve row-level predictions for paired comparisons after training.
    save_json(BASELINE_RUN / f"{name}_predictions.json", [
        {
            "id": str(row["id"]),
            "word": row["word"],
            "label": int(label),
            "prediction": int(prediction),
            **state["scores"][name][str(row["id"])],
        }
        for row, label, prediction in zip(rows, y, predictions)
    ])

    print(
        f"{name}: F1={report['f1']:.4f}, "
        f"precision={report['precision']:.4f}, "
        f"recall={report['recall']:.4f}, "
        f"AUC={report['auc']:.4f}"
    )
    print(
        f"  Examples with Yes/No mass below 0.90: "
        f"{report['yes_no_mass_below_0_90']}"
    )

save_json(BASELINE_RUN / "baseline_metrics.json", results)
print(f"\nCOMPLETE: {total} scores saved")
print(f"Recorded inference time: {state['inference_seconds'] / 60:.1f} minutes")
print("Saved:", BASELINE_RUN)
print("No model training or evaluation-threshold tuning performed.")

In [ ]:
import json, hashlib, time, numpy as np, torch
from tqdm.auto import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from project_utils import save_json

def section(t):
    print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78)

for name in ["gemma_model", "gemma_processor", "gemma_tokenizer", "YES_ID", "NO_ID", "BASELINE_RUN", "PROJECT"]:
    assert name in globals(), f"'{name}' missing: run the model-loading cell first."

# ---------------------------------------------------------------------------
section("1. LOAD AND VERIFY THE RUSSIAN SELECTION SET (used ONLY to choose the cutoff)")
PILOT_DIR = PROJECT / "outputs/runs/step07_pilot_dataset/20261105T223146Z_1dd5a63b"
pm = json.loads((PILOT_DIR / "manifest.json").read_text())
SEL = PILOT_DIR / "pilot_selection_ru_dev.jsonl"
assert hashlib.sha256(SEL.read_bytes()).hexdigest() == pm["sha256"]["pilot_selection_ru_dev.jsonl"], "Selection file changed."
sel_rows = [json.loads(line) for line in SEL.open(encoding="utf-8") if line.strip()]
assert len({r["id"] for r in sel_rows}) == len(sel_rows) == 2946
print(f"Selection file verified: {len(sel_rows)} examples (hash PASS)")

# Reviewer check: selection examples belong only to the reserved selection words.
split_file = PROJECT / "outputs/runs/step07_frozen_splits/20261105T220410Z_0c2b5b52/splits.json"
word_lists = json.loads(split_file.read_text())["word_lists"]
normalize = lambda w: w.strip().casefold().replace("ё", "е")
selection_words = {normalize(r["word"]) for r in sel_rows}
reserved_words = {normalize(w) for w in word_lists["AXOLOTL_RU_selection"]}
excluded_words = {normalize(w) for key in ("AXOLOTL_RU_train", "AXOLOTL_RU_validation") for w in word_lists[key]}
assert selection_words <= reserved_words, "Unexpected selection words."
assert not selection_words & excluded_words, "Selection overlaps train/evaluation."
assert {int(r["label"]) for r in sel_rows} == {0, 1}
gemma_model.eval()
print("Selection word separation: PASS")

# ---------------------------------------------------------------------------
section("2. SCORE THE SELECTION SET (resumable)")
SEL_CK = BASELINE_RUN / "selection_scores.json"
sel_scores = json.loads(SEL_CK.read_text()) if SEL_CK.exists() else {}
pending = [r for r in sel_rows if r["id"] not in sel_scores]
print(f"Already scored: {len(sel_scores)} | remaining: {len(pending)}")
gemma_tokenizer.padding_side = "left"
t0 = time.perf_counter()
try:
    with tqdm(total=len(pending), desc="RU selection", unit="example") as bar:
        for i in range(0, len(pending), 2):
            batch = pending[i:i + 2]
            inputs = gemma_processor.apply_chat_template(
                [r["messages"] for r in batch], tokenize=True, return_dict=True, return_tensors="pt",
                add_generation_prompt=True, enable_thinking=False,
                processor_kwargs={"padding": True, "truncation": False}).to("cuda")
            assert inputs["attention_mask"][:, -1].all().item(), "Padding is not on the left."
            with torch.inference_mode():
                logits = gemma_model(**inputs, use_cache=False, logits_to_keep=1).logits[:, -1, :].float()
            assert torch.isfinite(logits).all()
            p = torch.softmax(logits, -1)
            for r, m, mass in zip(batch, (logits[:, YES_ID] - logits[:, NO_ID]).tolist(),
                                  (p[:, YES_ID] + p[:, NO_ID]).tolist()):
                sel_scores[r["id"]] = {"yes_minus_no": float(m), "yes_no_mass": float(mass)}
            bar.update(len(batch))
            if (i // 2 + 1) % 50 == 0:
                save_json(SEL_CK, sel_scores)
finally:
    save_json(SEL_CK, sel_scores)
assert set(sel_scores) == {r["id"] for r in sel_rows}, "Incomplete: re-run this cell to resume."
low = sum(v["yes_no_mass"] < 0.9 for v in sel_scores.values())
print(f"Scored in {time.perf_counter() - t0:.0f}s | Yes/No mass below 0.90: {low}")

# ---------------------------------------------------------------------------
section("3. CHOOSE THE CUTOFF ON THE SELECTION SET (no evaluation labels used)")
y_sel = np.array([int(r["label"]) for r in sel_rows])
s_sel = np.array([sel_scores[r["id"]]["yes_minus_no"] for r in sel_rows])
# Reviewer check: scores are finite and inside the search range.
assert np.isfinite(s_sel).all(), "Non-finite selection scores."
assert s_sel.min() > -40 and s_sel.max() < 40, "Scores exceed the threshold search range; stop and report the range."
grid = [round(float(x), 2) for x in np.arange(-40, 40.001, 0.25)]
f1_at = lambda y, s, t: f1_score(y, (s >= t).astype(int), zero_division=0)
T_SEL = max(grid, key=lambda t: (f1_at(y_sel, s_sel, t), -t))   # best F1; ties -> lower cutoff
print(f"Selection-chosen cutoff : {T_SEL}   (selection F1 {f1_at(y_sel, s_sel, T_SEL):.4f}, "
      f"selection AUC {roc_auc_score(y_sel, s_sel):.4f})")
print(f"Frozen Swedish cutoff   : 10.5    (selection F1 {f1_at(y_sel, s_sel, 10.5):.4f})")

# ---------------------------------------------------------------------------
section("4. UNCHANGED GEMMA ON THE EVALUATION SETS")
print("PRIMARY = selection-chosen cutoff (same procedure will be used for the fine-tuned model).")
print("REFERENCE = frozen 10.5. Do not choose between them based on which evaluation F1 looks better.\n")
print(f"{'dataset':<26}{'cutoff':<22}{'F1':>7}{'Prec':>7}{'Recall':>8}{'AUC':>7}")
out = {"selection_cutoff": T_SEL, "selection_f1": float(f1_at(y_sel, s_sel, T_SEL)),
       "selection_auc": float(roc_auc_score(y_sel, s_sel)), "datasets": {},
       "policy": "Primary = selection-chosen cutoff; reference = frozen 10.5; no choice based on evaluation F1."}
for name in ["EN_wic_dev", "RU_axolotl_test_derived"]:
    preds = json.loads((BASELINE_RUN / f"{name}_predictions.json").read_text())
    y = np.array([p["label"] for p in preds]); s = np.array([p["yes_minus_no"] for p in preds])
    auc = float(roc_auc_score(y, s))
    out["datasets"][name] = {}
    for label, t in [(f"PRIMARY selection {T_SEL}", T_SEL), ("reference 10.5", 10.5)]:
        pr = (s >= t).astype(int)
        m = dict(cutoff=t, f1=float(f1_score(y, pr, zero_division=0)),
                 precision=float(precision_score(y, pr, zero_division=0)),
                 recall=float(recall_score(y, pr, zero_division=0)), auc=auc)
        out["datasets"][name][label] = m
        print(f"{name:<26}{label:<22}{m['f1']:>7.4f}{m['precision']:>7.4f}{m['recall']:>8.4f}{auc:>7.4f}")
save_json(BASELINE_RUN / "baseline_metrics_selection_cutoff.json", out)

section("SUMMARY")
print("'Before' numbers for unchanged Gemma are complete:")
print("  - English and Russian: PRIMARY row (selection-chosen cutoff) is what the fine-tuned model is compared against")
print("  - Swedish dev: 0.6782 (existing nested-CV result; same model, revision and prompt)")
print(f"Saved: {BASELINE_RUN}")
print("Next: QLoRA training setup and a short trial run.")

In [ ]:
import sys, subprocess, importlib
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)
importlib.invalidate_caches()
from peft.import_utils import is_torchao_available
is_torchao_available.cache_clear()
print("torchao seen by peft:", is_torchao_available(), "(must be False)")
assert is_torchao_available() is False, "STOP: paste this output."

import torch.nn as nn
lora_parts = [n for n, _ in gemma_model.named_modules() if "lora_" in n]
print("LoRA pieces inside Gemma:", len(lora_parts), "(must be 0)")
assert not lora_parts and not hasattr(gemma_model, "peft_config"), "STOP: restart session and reload Gemma."
print("PASS: fix applied, Gemma unchanged. Now re-run the trial cell.")

In [ ]:
import sys, subprocess, json, time, math, hashlib, random
from collections import Counter
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft"], check=True)
import torch, torch.nn as nn, torch.nn.functional as F
import peft
from peft import LoraConfig, get_peft_model
from project_utils import new_run, save_json

def section(t):
    print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78)

for name in ["gemma_model", "gemma_processor", "gemma_tokenizer", "YES_ID", "NO_ID", "PROJECT", "sel_rows"]:
    assert name in globals(), f"'{name}' missing: run the model-loading and selection cells first."
assert not isinstance(gemma_model, peft.PeftModel) and not hasattr(gemma_model, "peft_config"), \
    "An adapter is already attached: Runtime > Restart session, reload Gemma, then run this cell."
TRIAL = new_run(PROJECT, "step09_lora_trial")

# ---------------------------------------------------------------------------
section("1. VERIFY THE FROZEN TRAINING DATA")
PILOT_DIR = PROJECT / "outputs/runs/step07_pilot_dataset/20261105T223146Z_1dd5a63b"
pm = json.loads((PILOT_DIR / "manifest.json").read_text())
TRAIN = PILOT_DIR / "pilot_train.jsonl"
assert hashlib.sha256(TRAIN.read_bytes()).hexdigest() == pm["sha256"]["pilot_train.jsonl"], "Training file changed."
train_rows = [json.loads(line) for line in TRAIN.open(encoding="utf-8") if line.strip()]
assert len(train_rows) == 8510
assert all(int(r["label"]) in (0, 1) for r in train_rows)
for r in train_rows:
    msgs = r["messages"]
    assert msgs and msgs[-1]["role"] == "user", "Training messages must end with the question, not the answer."
    assert all(m["role"] in {"system", "user"} for m in msgs), \
        "Assistant messages found. Stop: inspect the training format before using this answer-token training loop."
train_rows = list(train_rows)
random.Random(2027).shuffle(train_rows)   # in-memory order only; the frozen file is unchanged
print(f"Training file hash: PASS | {len(train_rows)} examples | languages {dict(Counter(r['lang'] for r in train_rows))} | "
      f"YES {sum(r['label'] for r in train_rows)} / NO {sum(1 - r['label'] for r in train_rows)}")
print("Prompt format check (no answers inside prompts): PASS | in-memory shuffle: seed 2027")

# ---------------------------------------------------------------------------
section("2. WHERE THE ADAPTER GOES: language-model attention layers only")
SUFFIXES = ("q_proj", "k_proj", "v_proj", "o_proj")
linear = [n for n, m in gemma_model.named_modules() if isinstance(m, nn.Linear) and n.rsplit(".", 1)[-1] in SUFFIXES]
lang_layers = [n for n in linear if "language_model" in n]
other_layers = [n for n in linear if "language_model" not in n]
print(f"Attention projection layers found : {len(linear)}")
print(f"  in the language model (used)    : {len(lang_layers)}   e.g. {lang_layers[:2]}")
print(f"  elsewhere, e.g. vision (skipped): {len(other_layers)}   e.g. {other_layers[:2]}")
assert lang_layers, "No language-model attention layers found: STOP and paste this output."

# ---------------------------------------------------------------------------
section("3. ATTACH A FRESH LoRA ADAPTER (bf16 base model, no 4-bit quantisation)")
torch.manual_seed(2027)
cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules=lang_layers, bias="none")
model = get_peft_model(gemma_model, cfg)
model.config.use_cache = False
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
assert all("lora_" in name for name, parameter in model.named_parameters() if parameter.requires_grad), \
    "Unexpected base-model parameters are trainable."
trainable = [p for p in model.parameters() if p.requires_grad]
n_tr, n_all = sum(p.numel() for p in trainable), sum(p.numel() for p in model.parameters())
print(f"PEFT {peft.__version__} | trainable parameters {n_tr / 1e6:.1f}M of {n_all / 1e9:.1f}B ({100 * n_tr / n_all:.3f}%)")
print(f"Adapter weight dtypes: {sorted({str(p.dtype) for p in trainable})}")
print("Only LoRA weights trainable: PASS")
assert 0 < n_tr < 0.01 * n_all, "Unexpected number of trainable parameters."

# ---------------------------------------------------------------------------
MICRO, ACCUM, STEPS, LR = 1, 16, 20, 1e-4
section(f"4. TRIAL TRAINING: {STEPS} steps x {MICRO * ACCUM} examples = {STEPS * MICRO * ACCUM} examples")
print("Loss = cross-entropy on the answer token only (Yes/No), same position the scoring reads.")
print("Batch accuracy = Yes-vs-No direction on TRAINING examples. It is NOT an evaluation result.\n")
opt = torch.optim.AdamW(trainable, lr=LR, weight_decay=0.0)
gemma_tokenizer.padding_side = "left"
model.train()
torch.cuda.reset_peak_memory_stats()
log, cursor, t0 = [], 0, time.perf_counter()
for step in range(1, STEPS + 1):
    for g in opt.param_groups:
        g["lr"] = LR * min(1.0, step / 5)          # 5-step warm-up
    opt.zero_grad(set_to_none=True)
    step_loss, correct, seen = 0.0, 0, 0
    for _ in range(ACCUM):
        batch = train_rows[cursor:cursor + MICRO]; cursor += MICRO
        inputs = gemma_processor.apply_chat_template(
            [r["messages"] for r in batch], tokenize=True, return_dict=True, return_tensors="pt",
            add_generation_prompt=True, enable_thinking=False,
            processor_kwargs={"padding": True, "truncation": False}).to("cuda")
        assert inputs["attention_mask"][:, -1].all().item(), "Padding is not on the left."
        labels = torch.tensor([int(r["label"]) for r in batch], device="cuda")
        targets = torch.where(labels == 1, torch.tensor(YES_ID, device="cuda"), torch.tensor(NO_ID, device="cuda"))
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            logits = model(**inputs, use_cache=False, logits_to_keep=1).logits[:, -1, :]
        loss = F.cross_entropy(logits.float(), targets)
        assert torch.isfinite(loss), "Non-finite loss: STOP."
        (loss / ACCUM).backward()
        step_loss += loss.item() / ACCUM
        correct += int(((logits[:, YES_ID] > logits[:, NO_ID]).long() == labels).sum()); seen += len(batch)
    grad_norm = float(torch.nn.utils.clip_grad_norm_(trainable, 1.0, error_if_nonfinite=True))
    opt.step()
    log.append(dict(step=step, loss=step_loss, batch_accuracy=correct / seen, grad_norm=grad_norm,
                    seconds=time.perf_counter() - t0))
    print(f"  step {step:>2}/{STEPS}  loss {step_loss:.4f}  batch accuracy {correct / seen:.2f}  "
          f"grad-norm {grad_norm:.3f}  elapsed {time.perf_counter() - t0:.0f}s")
elapsed = time.perf_counter() - t0
peak = torch.cuda.max_memory_allocated() / 2**30

# ---------------------------------------------------------------------------
section("5. DOES THE ADAPTER CHANGE GEMMA'S SCORES? (40 selection examples, adapter on vs off)")
model.eval()
probe = sel_rows[:40]
def margins(m):
    out = []
    with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        for i in range(0, len(probe), 4):
            inputs = gemma_processor.apply_chat_template(
                [r["messages"] for r in probe[i:i + 4]], tokenize=True, return_dict=True, return_tensors="pt",
                add_generation_prompt=True, enable_thinking=False,
                processor_kwargs={"padding": True, "truncation": False}).to("cuda")
            assert inputs["attention_mask"][:, -1].all().item(), "Padding is not on the left."
            lg = m(**inputs, use_cache=False, logits_to_keep=1).logits[:, -1, :].float()
            out += (lg[:, YES_ID] - lg[:, NO_ID]).tolist()
    return torch.tensor(out)
with_adapter = margins(model)
with model.disable_adapter():
    without_adapter = margins(model)
delta = (with_adapter - without_adapter).abs()
print(f"Mean |change in Yes-No score|: {delta.mean():.3f} | max {delta.max():.3f}")
print("PASS: adapter is active" if delta.mean() > 1e-3 else "WARNING: adapter has no visible effect")

# ---------------------------------------------------------------------------
full_steps = math.ceil(len(train_rows) / (MICRO * ACCUM))
per_step = elapsed / STEPS
model.save_pretrained(str(TRIAL / "adapter_trial"))
save_json(TRIAL / "trial_log.json", {
    "config": dict(r=16, alpha=32, dropout=0.05, target="language-model q/k/v/o", lr=LR, micro_batch=MICRO,
                   grad_accum=ACCUM, steps=STEPS, warmup_steps=5, base="bf16, no quantisation", autocast="bf16",
                   peft=peft.__version__, loss="cross-entropy on answer token", in_memory_shuffle_seed=2027,
                   grad_clip=1.0, error_if_nonfinite=True),
    "log": log, "peak_gpu_gib": peak, "seconds_per_step": per_step,
    "adapter_effect_mean_abs_delta": float(delta.mean()), "note": "Trial adapter is NOT used for evaluation."})

section("SUMMARY")
print(f"Loss and gradients finite : PASS (all {STEPS} steps)")
print(f"Loss: first step {log[0]['loss']:.4f} -> last step {log[-1]['loss']:.4f} (short trial: expect noise)")
print(f"Peak GPU memory           : {peak:.1f} GiB of 80")
print(f"Speed                     : {per_step:.1f}s per step of {MICRO * ACCUM} examples")
print(f"Full run estimate         : {full_steps} steps (1 pass over 8,510) = about {full_steps * per_step / 60:.0f} minutes")
print(f"Saved: {TRIAL}")
print("The trial adapter is NOT the final model. The full run will start from a fresh adapter.")

In [ ]:
"""Run in the SAME Colab runtime as the completed BF16 LoRA trial.

One epoch, fresh adapter; checkpoint/cutoff selection uses RU selection only.
Resume by rerunning this file in that runtime. After a runtime reset, reload
the pinned BF16 base and processor first, then rerun this file.
"""
import gc
import hashlib
import json
import math
import random
import shutil
import time
import uuid
from contextlib import nullcontext
from importlib.metadata import version
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from peft import LoraConfig, PeftModel, get_peft_model
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from tqdm.auto import tqdm
from project_utils import new_run, save_json, frozen, save_evaluation

PROJECT = Path('/content/drive/MyDrive/semeval2027/semeval2027_colab')
PILOT = PROJECT / 'outputs/runs/step07_pilot_dataset/20261105T223146Z_1dd5a63b'
EVAL = PROJECT / 'outputs/runs/step07_pilot_evaluation/20261105T223700Z_5259f45a'
BASELINE = PROJECT / 'outputs/runs/step08_gemma_unchanged_baseline/20261105T224131Z_63bba2d9'
SWEDISH_REF = PROJECT / 'outputs/runs/step04_gemma4_evaluation/20261105T170910Z_b3fc303a'
MODEL_ID = 'google/gemma-4-31B-it'
REVISION = '842da3794eaa0b77d5f08bae87a17459d91ff475'
POINTER = PILOT / 'full_bf16_lora_run_pointer.json'
SEED, EFFECTIVE_BATCH, LR = 2027, 16, 1e-4
SAVE_EVERY = 25
GRID = tuple([-1000.0] + np.arange(-100, 100.001, .25).tolist() + [1000.0])  # widened before any adapted scores exist

def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def read_rows(path):
    with Path(path).open(encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

def normalized(w):
    return w.strip().casefold().replace('ё', 'е')

def metrics(rows, margins, threshold):
    y = np.asarray([int(r['label']) for r in rows])
    s = np.asarray(margins, dtype=float)
    assert np.isfinite(s).all() and set(y) == {0, 1}
    pred = (s >= threshold).astype(int)
    return dict(n=len(rows), threshold=float(threshold),
                f1=float(f1_score(y, pred, zero_division=0)),
                precision=float(precision_score(y, pred, zero_division=0)),
                recall=float(recall_score(y, pred, zero_division=0)),
                auc=float(roc_auc_score(y, s)))

def selection_metrics(rows, margins):
    assert np.isfinite(margins).all() and np.max(np.abs(margins)) < 100, (
        'Selection scores outside the predeclared grid; checkpoint saved. '
        'Stop and report the score range before changing the protocol.')
    y = np.asarray([int(r['label']) for r in rows])
    s = np.asarray(margins)
    threshold = max(GRID, key=lambda t: (
        f1_score(y, s >= t, zero_division=0), -t))
    return metrics(rows, s, threshold)

# Verify files before touching the model. No evaluation scores are used here.
assert torch.cuda.is_available(), 'Reload the pinned base on the A100 first.'
for name in ('gemma_processor', 'gemma_tokenizer'):
    assert name in globals(), f'{name} missing; reload the processor first.'
pm = json.loads((PILOT / 'manifest.json').read_text())
train_path, select_path = PILOT / 'pilot_train.jsonl', PILOT / 'pilot_selection_ru_dev.jsonl'
for path in (train_path, select_path):
    assert digest(path) == pm['sha256'][path.name], f'File changed: {path}'
training, selection = read_rows(train_path), read_rows(select_path)
assert len(training) == 8510 and len(selection) == 2946
for rows in (training, selection):
    assert len({str(r['id']) for r in rows}) == len(rows)
    for r in rows:
        assert int(r['label']) in (0, 1)
        assert r['messages'] and r['messages'][-1]['role'] == 'user'
        assert all(m['role'] in ('system', 'user') for m in r['messages'])

split_path = PROJECT / 'outputs/runs/step07_frozen_splits/20261105T220410Z_0c2b5b52/splits.json'
wl = json.loads(split_path.read_text())['word_lists']
selection_words = {normalized(r['word']) for r in selection}
assert selection_words <= {normalized(w) for w in wl['AXOLOTL_RU_selection']}
assert not selection_words & {
    normalized(w) for key in ('AXOLOTL_RU_train', 'AXOLOTL_RU_validation') for w in wl[key]}
sv_words = {'dumpa', 'exportera', 'fru', 'fröken', 'förort', 'herre', 'kvinna', 'motiv', 'skär', 'suga'}
for r in training:
    lang = r['lang']
    assert lang in ('RU', 'EN'), f'Unexpected pilot language: {lang}'
    key = 'AXOLOTL_RU_train' if lang == 'RU' else 'WiC_EN_train'
    assert normalized(r['word']) in {normalized(w) for w in wl[key]}
    assert normalized(r['word']) not in sv_words

eval_manifest = json.loads((EVAL / 'manifest.json').read_text())
for filename, expected in eval_manifest['output_sha256'].items():
    assert digest(EVAL / filename) == expected, f'Evaluation file changed: {filename}'
base_calibration = json.loads((BASELINE / 'baseline_metrics_selection_cutoff.json').read_text())
base_selection_scores = json.loads((BASELINE / 'selection_scores.json').read_text())
assert set(base_selection_scores) == {str(r['id']) for r in selection}
base_selection = selection_metrics(selection, [
    base_selection_scores[str(r['id'])]['yes_minus_no'] for r in selection])
assert base_selection['threshold'] == base_calibration['selection_cutoff'] == 9.0

# Pre-flight: everything the LAST stage needs is checked NOW, before hours of training.
from semeval_task3.data import load_dataset
from semeval_task3.cv import Candidate, nested_word_cv
from semeval_task3.evaluation import score, read_submission, word_bootstrap
_sv = load_dataset(PROJECT / 'work/data/dev', with_gold=True)
assert {u.language for u in _sv.usages} == {'SV'} and len({u.word for u in _sv.usages}) == 10
for _u in _sv.usages:
    _u.marked_text()
_orig = score(_sv.require_gold(), read_submission(SWEDISH_REF / 'gemma_raw.zip', _sv.keys))
assert abs(_orig['average'] - 0.6782392862783919) < 1e-6, f"Swedish reference F1 changed: {_orig['average']}"
for _name in ('EN_wic_dev', 'RU_axolotl_test_derived'):
    _old = json.loads((BASELINE / f'{_name}_predictions.json').read_text())
    assert all({'id', 'label', 'yes_minus_no'} <= set(p) for p in _old), f'Unexpected format: {_name}'
print(f"PRE-FLIGHT PASS: data hashes, word lists, baseline cutoff 9.0, Swedish data ({len(_sv.usages)} usages, "
      f"reference F1 {_orig['average']:.4f}), baseline prediction files.", flush=True)
del _sv, _orig, _old

random.Random(SEED).shuffle(training)
TOTAL = math.ceil(len(training) / EFFECTIVE_BATCH)
MILESTONES = sorted({math.ceil(TOTAL * .25), math.ceil(TOTAL * .5), TOTAL})
WARMUP = math.ceil(.05 * TOTAL)
config = dict(model=MODEL_ID, revision=REVISION, seed=SEED, epochs=1,
              examples=len(training), micro_batch=1, effective_batch=EFFECTIVE_BATCH,
              learning_rate=LR, warmup_steps=WARMUP, schedule='linear warmup then linear decay',
              steps=TOTAL, milestones=MILESTONES, r=16, alpha=32, dropout=.05,
              target='language-model q/k/v/o', base_precision='BF16, no quantization',
              loss='full-vocabulary cross entropy at next answer token',
              train_sha256=digest(train_path), selection_sha256=digest(select_path),
              split_sha256=digest(split_path), evaluation_sha256=eval_manifest['output_sha256'],
              baseline_selection_sha256=digest(BASELINE / 'selection_scores.json'),
              cutoff_grid=list(GRID), checkpoint_selection='highest RU selection pooled F1; ties prefer earlier step, including unchanged baseline',
              packages={p: version(p) for p in ('torch', 'transformers', 'peft')})

if POINTER.exists():
    FULL_RUN = Path(json.loads(POINTER.read_text())['run_directory'])
    assert json.loads((FULL_RUN / 'config.json').read_text()) == config, (
        'Run configuration changed. Do not overwrite or resume this run with different settings.')
else:
    FULL_RUN = new_run(PROJECT, 'step09_lora_full_pilot')
    save_json(FULL_RUN / 'config.json', config)
    save_json(POINTER, {'run_directory': str(FULL_RUN)})
print('RUN:', FULL_RUN, flush=True)
print(f'{TOTAL} updates; selection checks at {MILESTONES}; warmup {WARMUP} updates.', flush=True)
print('Resumable checkpoints every 25 updates. Evaluation labels do not choose checkpoints.', flush=True)

if (FULL_RUN / 'COMPLETE.json').exists():
    print('This run is already complete. Read:', FULL_RUN / 'COMPLETE.json')
else:
    # unload() removes adapters WITHOUT merging their changes into the base.
    current = globals().get('full_model')
    if current is None:
        current = globals().get('model')
    if current is None:
        current = globals().get('gemma_model')
    assert current is not None, 'Reload the pinned BF16 base first.'
    assert not getattr(current, 'is_loaded_in_4bit', False), 'This script expects the tested BF16 base.'
    base = current.unload() if isinstance(current, PeftModel) else current
    assert base.config._name_or_path == MODEL_ID, 'Unexpected base model.'
    assert not any('lora_' in n for n, _ in base.named_parameters())
    # Release optimizer/adapter references left by the trial or a previous attempt.
    for name in ('opt', 'optimizer', 'full_optimizer', 'trainable', 'model', 'full_model', 'gemma_model'):
        globals().pop(name, None)
    del current
    gc.collect()
    torch.cuda.empty_cache()
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    random.seed(SEED)
    np.random.seed(SEED)
    gemma_tokenizer.padding_side = 'left'
    YES_ID = gemma_tokenizer.encode('Yes', add_special_tokens=False)[0]
    NO_ID = gemma_tokenizer.encode('No', add_special_tokens=False)[0]
    assert gemma_tokenizer.encode('Yes', add_special_tokens=False) == [10784]
    assert gemma_tokenizer.encode('No', add_special_tokens=False) == [3771]
    targets = [n for n, m in base.named_modules()
               if isinstance(m, torch.nn.Linear) and 'language_model' in n.split('.')
               and n.split('.')[-1] in ('q_proj', 'k_proj', 'v_proj', 'o_proj')]
    assert len(targets) == 230, f'Unexpected projection count: {len(targets)}'

    latest_file = FULL_RUN / 'latest.json'
    resume_state = None
    if latest_file.exists():
        latest = Path(json.loads(latest_file.read_text())['directory'])
        full_model = PeftModel.from_pretrained(base, str(latest / 'adapter'), is_trainable=True)
        # Only load optimizer/RNG state created by this script in your own run.
        resume_state = torch.load(latest / 'training_state.pt', map_location='cpu', weights_only=False)
        step_done, train_log = resume_state['step'], resume_state['log']
        print(f'Resuming after update {step_done}.', flush=True)
    else:
        full_model = get_peft_model(base, LoraConfig(
            r=16, lora_alpha=32, lora_dropout=.05, target_modules=targets, bias='none'))
        step_done, train_log = 0, []
        assert all(torch.count_nonzero(p).item() == 0 for n, p in full_model.named_parameters() if 'lora_B' in n)
        print('Fresh adapter: PASS. Trial weights discarded without merging.', flush=True)
    full_model.config.use_cache = False
    full_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
    params = [p for p in full_model.parameters() if p.requires_grad]
    assert params and all('lora_' in n for n, p in full_model.named_parameters() if p.requires_grad)
    full_optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)
    if resume_state:
        full_optimizer.load_state_dict(resume_state['optimizer'])
        random.setstate(resume_state['python_rng'])
        np.random.set_state(resume_state['numpy_rng'])
        torch.set_rng_state(resume_state['torch_rng'])
        torch.cuda.set_rng_state_all(resume_state['cuda_rng'])
        del resume_state

    def save_resume(step):
        folder = FULL_RUN / f'resume_{step:04d}_{uuid.uuid4().hex[:8]}'
        folder.mkdir()
        full_model.save_pretrained(folder / 'adapter', safe_serialization=True)
        torch.save(dict(step=step, log=train_log, optimizer=full_optimizer.state_dict(),
                        python_rng=random.getstate(), numpy_rng=np.random.get_state(),
                        torch_rng=torch.get_rng_state(), cuda_rng=torch.cuda.get_rng_state_all()),
                   folder / 'training_state.pt')
        previous = json.loads(latest_file.read_text())['directory'] if latest_file.exists() else None
        save_json(latest_file, {'directory': str(folder), 'step': step})
        # Delete only the previous rolling checkpoint belonging to this run.
        if previous:
            previous = Path(previous)
            if previous.parent == FULL_RUN and previous.name.startswith('resume_'):
                shutil.rmtree(previous)
        save_json(FULL_RUN / 'training_log.json', train_log)

    def score_rows(rows, destination, model_tag, disable=False):
        """Resumable inference; labels never enter the model."""
        identity = hashlib.sha256(json.dumps(
            [(str(r['id']), r['messages']) for r in rows],
            ensure_ascii=False, sort_keys=True).encode()).hexdigest()
        if destination.exists():
            saved = json.loads(destination.read_text())
            assert saved['input_sha256'] == identity and saved['model_tag'] == model_tag
        else:
            saved = dict(input_sha256=identity, model_tag=model_tag, scores={})
        assert set(saved['scores']) <= {str(r['id']) for r in rows}
        pending = [r for r in rows if str(r['id']) not in saved['scores']]
        full_model.eval()
        context = full_model.disable_adapter() if disable else nullcontext()
        try:
            with context, torch.inference_mode(), torch.autocast('cuda', dtype=torch.bfloat16):
                for i in tqdm(range(0, len(pending), 2), desc=model_tag):
                    batch = pending[i:i+2]
                    inputs = gemma_processor.apply_chat_template(
                        [r['messages'] for r in batch], tokenize=True, return_dict=True,
                        return_tensors='pt', add_generation_prompt=True, enable_thinking=False,
                        processor_kwargs={'padding': True, 'truncation': False}).to('cuda')
                    assert inputs['attention_mask'][:, -1].all()
                    logits = full_model(**inputs, use_cache=False, logits_to_keep=1).logits[:, -1, :].float()
                    assert torch.isfinite(logits).all()
                    margins = (logits[:, YES_ID] - logits[:, NO_ID]).tolist()
                    lp = logits.log_softmax(-1)
                    masses = (lp[:, YES_ID].exp() + lp[:, NO_ID].exp()).tolist()
                    for r, margin, mass in zip(batch, margins, masses):
                        saved['scores'][str(r['id'])] = dict(margin=float(margin), yes_no_mass=float(mass))
                    del inputs, logits, lp
                    if (i // 2 + 1) % 25 == 0:
                        save_json(destination, saved)
        finally:
            save_json(destination, saved)
        assert set(saved['scores']) == {str(r['id']) for r in rows}
        return [saved['scores'][str(r['id'])]['margin'] for r in rows]

    def milestone(step):
        folder = FULL_RUN / f'checkpoint_{step:04d}'
        folder.mkdir(exist_ok=True)
        if not (folder / 'adapter' / 'adapter_config.json').exists():
            full_model.save_pretrained(folder / 'adapter', safe_serialization=True)
        result_path = folder / 'selection_metrics.json'
        if not result_path.exists():
            scores = score_rows(selection, folder / 'selection_scores.json', f'selection_step_{step}')
            result = selection_metrics(selection, scores)
            save_json(result_path, dict(step=step, **result))
        result = json.loads(result_path.read_text())
        print(f"SELECTION step {step}: F1={result['f1']:.4f}, cutoff={result['threshold']}, AUC={result['auc']:.4f}", flush=True)
        full_model.train()

    # A milestone interrupted during scoring is completed before training resumes.
    if step_done in MILESTONES:
        milestone(step_done)
    full_model.train()
    torch.cuda.reset_peak_memory_stats()
    session_start = time.perf_counter()
    for step in range(step_done + 1, TOTAL + 1):
        group = training[(step-1)*EFFECTIVE_BATCH:step*EFFECTIVE_BATCH]
        assert group
        scale = step / WARMUP if step <= WARMUP else (TOTAL-step+1) / (TOTAL-WARMUP)
        for pg in full_optimizer.param_groups:
            pg['lr'] = LR * scale
        full_optimizer.zero_grad(set_to_none=True)
        loss_sum, correct = 0.0, 0
        for r in group:
            inputs = gemma_processor.apply_chat_template(
                [r['messages']], tokenize=True, return_dict=True, return_tensors='pt',
                add_generation_prompt=True, enable_thinking=False,
                processor_kwargs={'padding': True, 'truncation': False}).to('cuda')
            assert inputs['attention_mask'][:, -1].all()
            answer = torch.tensor([YES_ID if int(r['label']) else NO_ID], device='cuda')
            with torch.autocast('cuda', dtype=torch.bfloat16):
                logits = full_model(**inputs, use_cache=False, logits_to_keep=1).logits[:, -1, :]
            loss = F.cross_entropy(logits.float(), answer)
            assert torch.isfinite(loss), 'Non-finite loss; resume from last saved checkpoint.'
            (loss / len(group)).backward()  # Correct also for the final partial batch.
            loss_sum += float(loss.detach())
            correct += int((logits[0, YES_ID] > logits[0, NO_ID]).item() == bool(int(r['label'])))
            del inputs, logits, loss, answer
        norm = float(torch.nn.utils.clip_grad_norm_(params, 1.0, error_if_nonfinite=True))
        full_optimizer.step()
        train_log.append(dict(step=step, loss=loss_sum/len(group), batch_accuracy=correct/len(group),
                              grad_norm=norm, lr=LR*scale, examples=len(group)))
        if step == 1 or step % 10 == 0 or step in MILESTONES:
            print(f'Update {step}/{TOTAL}: loss={loss_sum/len(group):.4f}, '
                  f'accuracy={correct/len(group):.3f}, elapsed this session='
                  f'{(time.perf_counter()-session_start)/60:.1f} min', flush=True)
        if step % SAVE_EVERY == 0 or step in MILESTONES:
            save_resume(step)
        if step in MILESTONES:
            milestone(step)

    # Select once using ONLY Russian selection results; baseline is eligible.
    candidates = [dict(step=0, **base_selection)] + [
        json.loads((FULL_RUN / f'checkpoint_{s:04d}/selection_metrics.json').read_text())
        for s in MILESTONES]
    best = max(candidates, key=lambda x: (x['f1'], -x['step']))
    save_json(FULL_RUN / 'selection_decision.json', dict(candidates=candidates, selected=best))
    print('\nSELECTED USING RUSSIAN SELECTION ONLY:', best, flush=True)
    full_optimizer.zero_grad(set_to_none=True)
    del full_optimizer, params
    gc.collect()
    torch.cuda.empty_cache()
    if best['step']:
        base = full_model.unload()
        full_model = PeftModel.from_pretrained(
            base, str(FULL_RUN / f"checkpoint_{best['step']:04d}" / 'adapter'), is_trainable=False)
    else:
        full_model = full_model.unload()
    full_model.eval()
    gemma_processor.save_pretrained(FULL_RUN / 'processor')

    # External evaluation: one chosen checkpoint; no further checkpoint selection.
    evaluation = {}
    def paired_word_ci(rows, new_scores, old_scores, new_t, old_t):
        words = sorted({r['word'] for r in rows})
        index = {w: i for i, w in enumerate(words)}
        counts = np.zeros((len(words), 6), dtype=np.int64)
        for r, a, b in zip(rows, new_scores, old_scores):
            y = int(r['label'])
            for offset, pred in ((0, a >= new_t), (3, b >= old_t)):
                counts[index[r['word']], offset:offset+3] += [int(y == 1 and pred), int(y == 0 and pred), int(y == 1 and not pred)]
        def f1(c, off):
            tp, fp, fn = c[off:off+3]
            return 2*tp / max(2*tp+fp+fn, 1)
        generator = np.random.default_rng(SEED)
        diffs = []
        for _ in range(5000):
            c = counts[generator.integers(0, len(words), len(words))].sum(0)
            diffs.append(f1(c, 0)-f1(c, 3))
        c = counts.sum(0)
        return dict(estimate=f1(c, 0)-f1(c, 3), lower=float(np.quantile(diffs, .025)),
                    upper=float(np.quantile(diffs, .975)), unit='word', draws=5000,
                    note='Paired word bootstrap of pooled F1; fixed selected models/cutoffs.')

    for name in ('EN_wic_dev', 'RU_axolotl_test_derived'):
        rows = read_rows(EVAL / f'{name}.jsonl')
        old = {str(r['id']): r for r in json.loads((BASELINE / f'{name}_predictions.json').read_text())}
        assert set(old) == {str(r['id']) for r in rows}
        for r in rows:
            assert int(r['label']) == int(old[str(r['id'])]['label'])
        old_scores = [old[str(r['id'])]['yes_minus_no'] for r in rows]
        scores = score_rows(rows, FULL_RUN / f'{name}_scores.json', f"chosen_{best['step']}_{name}") if best['step'] else old_scores
        evaluation[name] = dict(
            original=metrics(rows, old_scores, base_selection['threshold']),
            selected=metrics(rows, scores, best['threshold']),
            paired_f1_difference=paired_word_ci(rows, scores, old_scores, best['threshold'], base_selection['threshold']))
        save_json(FULL_RUN / 'external_evaluation.json', evaluation)
        print(name, json.dumps(evaluation[name], indent=2), flush=True)

    # Swedish development: reproduce the established nested word-CV protocol.
    if best['step']:
        from semeval_task3.data import load_dataset
        from semeval_task3.cv import Candidate, nested_word_cv
        from semeval_task3.evaluation import score, read_submission, word_bootstrap
        sv = load_dataset(PROJECT / 'work/data/dev', with_gold=True)
        sv_rows = []
        sv_keys = []
        for u in sv.usages:
            assert u.language == 'SV'
            sv_keys.append(u.key)
            sv_rows.append(dict(id='|'.join(u.key), messages=[
                dict(role='system', content='You are an expert lexicographer of historical Swedish.'),
                dict(role='user', content=(f'Target word (lemma): {u.word}\n'
                    f'Sense definition (Swedish): {u.definition}\n'
                    f'Sentence from {u.year}: {u.marked_text()}\n\n'
                    'Does the word marked with <t> and </t> in the sentence express the sense given by the definition? Answer only Yes or No.'))]))
        sv_scores = score_rows(sv_rows, FULL_RUN / 'SV_scores.json', f"chosen_{best['step']}_SV")
        sv_map = dict(zip(sv_keys, sv_scores))
        assert np.max(np.abs(sv_scores)) < 100, 'Swedish margins outside grid; scores saved, report range.'
        result = nested_word_cv(sv, [Candidate('lora_selected', frozen(sv_map), GRID)])
        save_evaluation(FULL_RUN, 'lora_selected_SV', result, sv, PROJECT, official=True)
        original = score(sv.require_gold(), read_submission(SWEDISH_REF / 'gemma_raw.zip', sv.keys))
        save_json(FULL_RUN / 'swedish_comparison.json', dict(
            original_f1=original['average'], selected_f1=result['report']['average'],
            paired_difference=word_bootstrap(result['report'], other=original),
            note='Repeatedly used Swedish development data; not an untouched test.'))
    else:
        save_json(FULL_RUN / 'swedish_comparison.json', dict(
            selected='unchanged baseline', original_f1=0.6782392862783919,
            selected_f1=0.6782392862783919, note='Existing Swedish nested-CV reference; no adapter selected.'))

    save_json(FULL_RUN / 'COMPLETE.json', dict(selected=best, run_directory=str(FULL_RUN),
        status='pilot complete; review external and Swedish results before adopting an adapter',
        external_results='external_evaluation.json', swedish_results='swedish_comparison.json'))
    print('\nSWEDISH COMPARISON:', (FULL_RUN / 'swedish_comparison.json').read_text(), flush=True)
    print('\nFULL PILOT COMPLETE. Saved:', FULL_RUN, flush=True)
    print('Paste selection_decision.json, external_evaluation.json and swedish_comparison.json summaries.')

In [ ]:
import sys, subprocess, json
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bcubed"], check=True)
import bcubed  # confirms the official scorer's dependency is now installed

def section(t):
    print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78)

for name in ("FULL_RUN", "result", "sv", "best", "evaluation", "SWEDISH_REF"):
    assert name in globals(), f"'{name}' missing: STOP, paste this output, do NOT re-run the training script."
assert (FULL_RUN / "SV_scores.json").exists(), "Swedish scores not saved: STOP and paste this output."

# ---------------------------------------------------------------------------
section("1. FINISH THE SWEDISH EVALUATION (official scorer, nested word CV)")
save_evaluation(FULL_RUN, "lora_selected_SV", result, sv, PROJECT, official=True)
original = score(sv.require_gold(), read_submission(SWEDISH_REF / "gemma_raw.zip", sv.keys))
assert abs(original["average"] - 0.6782392862783919) < 1e-6, "Original Swedish reference changed."
paired = word_bootstrap(result["report"], other=original)
save_json(FULL_RUN / "swedish_comparison.json", dict(
    original_f1=original["average"], selected_f1=result["report"]["average"], paired_difference=paired,
    note="Repeatedly used Swedish development data; not an untouched test."))
save_json(FULL_RUN / "COMPLETE.json", dict(selected=best, run_directory=str(FULL_RUN),
    status="pilot complete; adoption decided in adoption_decision.json",
    external_results="external_evaluation.json", swedish_results="swedish_comparison.json"))
print("Official scorer agreed with our scorer, and the results are saved.")

# ---------------------------------------------------------------------------
section("2. SWEDISH PER WORD: original Gemma vs trained Gemma (held-out cutoffs)")
old_w = {r["word"]: r["f1"] for r in original["per_word"]}
new_w = {r["word"]: r["f1"] for r in result["report"]["per_word"]}
cut = {s["test_group"][1]: s["threshold"] for s in result["selections"]}
assert old_w.keys() == new_w.keys() and len(old_w) == 10
better = 0
print(f"{'word':<11}{'original':>10}{'trained':>10}{'change':>10}{'cutoff used':>13}   verdict")
for w in sorted(old_w):
    d = new_w[w] - old_w[w]
    better += d > 1e-12
    verdict = "better" if d > 1e-12 else ("same" if abs(d) <= 1e-12 else "worse")
    print(f"{w:<11}{old_w[w]:>10.3f}{new_w[w]:>10.3f}{d:>+10.3f}{cut[w]:>13.2f}   {verdict}")
print(f"\nSwedish F1: original {original['average']:.4f} -> trained {result['report']['average']:.4f} "
      f"(change {paired['estimate']:+.4f}, 95% range {paired['lower']:+.4f} to {paired['upper']:+.4f})")

# ---------------------------------------------------------------------------
section("3. THE PRE-REGISTERED ADOPTION RULE")
need = original["average"] + 0.02
r1a = result["report"]["average"] >= need
r1b = better >= 6
auc_change = {n: evaluation[n]["selected"]["auc"] - evaluation[n]["original"]["auc"] for n in evaluation}
r2 = all(v >= -0.02 for v in auc_change.values())
decision_file = json.loads((FULL_RUN / "selection_decision.json").read_text())
r3 = decision_file["selected"]["step"] == best["step"]
print(f"Rule 1a  Swedish F1 >= {need:.4f}            : {result['report']['average']:.4f}  -> {'PASS' if r1a else 'FAIL'}")
print(f"Rule 1b  better on >= 6 of 10 Swedish words  : {better}/10    -> {'PASS' if r1b else 'FAIL'}")
for n, v in auc_change.items():
    print(f"Rule 2   {n:<24} AUC change     : {v:+.4f} -> {'PASS' if v >= -0.02 else 'FAIL'}")
print(f"Rule 3   checkpoint chosen on RU selection only: step {best['step']} "
      f"(candidates: {[(c['step'], round(c['f1'], 4)) for c in decision_file['candidates']]}) -> {'PASS' if r3 else 'FAIL'}")
adopt = r1a and r1b and r2 and r3
save_json(FULL_RUN / "adoption_decision.json", dict(
    rule_1a=bool(r1a), rule_1b=bool(r1b), words_better=int(better), rule_2=bool(r2), auc_change=auc_change,
    rule_3=bool(r3), adopt=bool(adopt), swedish_original=original["average"],
    swedish_trained=result["report"]["average"]))

section("DECISION")
if adopt:
    print("ADOPT: trained Gemma (step", best["step"], ") becomes the primary system.")
else:
    print("KEEP ORIGINAL: unchanged Gemma (Swedish 0.678) stays the primary system.")
    print("The fine-tuning pilot is recorded as a documented result for the paper.")
print("Saved:", FULL_RUN / "adoption_decision.json")

In [ ]:
import hashlib
from collections import defaultdict
from project_utils import read_scores, save_json

def section(t):
    print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78)

for name in ("sv", "sv_map", "GRID", "FULL_RUN", "result", "original"):
    assert name in globals(), f"'{name}' missing: STOP and paste this output."
gold = sv.require_gold()
orig_scores = read_scores(PROJECT / "outputs/runs/step04_gemma4_31b/20261105T164154Z_48096e67/gemma_raw_scores.json", sv)
assert set(sv_map) == set(orig_scores) == sv.keys

def by_word(scores):
    d = defaultdict(list)
    for u in sv.usages:
        d[u.word].append((scores[u.key], gold[u.key]))
    return d

def f1(pairs, t):
    tp = sum(1 for s, y in pairs if s >= t and y == 1)
    fp = sum(1 for s, y in pairs if s >= t and y == 0)
    fn = sum(1 for s, y in pairs if s < t and y == 1)
    return 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else 0.0

def mean_f1(bw, t):
    return sum(f1(p, t) for p in bw.values()) / len(bw)

def oracle(bw):
    return sum(max(f1(p, t) for t in sorted({s for s, _ in p}) + [float("inf")]) for p in bw.values()) / len(bw)

section("1. HOW TUNING ON VISIBLE DEV LABELS INFLATES THE SAME SCORES (Swedish dev, 10 words)")
nested = {"Original Gemma": original["average"], "Trained Gemma": result["report"]["average"]}
fitted = {}
print(f"{'':<16}{'A. honest held-out':>20}{'C. one cutoff fit on dev':>27}{'D. per-word cutoffs':>22}")
for label, scores in (("Original Gemma", orig_scores), ("Trained Gemma", sv_map)):
    bw = by_word(scores)
    t = max(GRID, key=lambda x: (mean_f1(bw, x), -x))
    fitted[label] = (t, mean_f1(bw, t))
    print(f"{label:<16}{nested[label]:>20.4f}{fitted[label][1]:>19.4f} (@{t:>5.2f}){oracle(bw):>22.4f}")
print("\nA = what we report (cutoff chosen without the held-out word's labels).")
print("C = what a single tuned cutoff gives on public dev labels (likely what many leaderboard entries show).")
print("D = a separate cutoff per word tuned on labels: NOT possible on the real test.")
print("Leaderboard top Subtask 2 Swedish entries: 0.76, 0.73, 0.73, 0.72")

section("2. THE EXPORTERA TIE-BREAK, EXACT VALUES")
ow = {r["word"]: r for r in original["per_word"]}
nw = {r["word"]: r for r in result["report"]["per_word"]}
for w in ("exportera",):
    print(f"{w}: original F1 {ow[w]['f1']:.6f} (tp {ow[w]['tp']}, fp {ow[w]['fp']}, fn {ow[w]['fn']}) | "
          f"trained F1 {nw[w]['f1']:.6f} (tp {nw[w]['tp']}, fp {nw[w]['fp']}, fn {nw[w]['fn']})")

section("3. FREEZE THE ADOPTED SYSTEM (same procedure as the 10.5 freeze)")
adapter_dir = FULL_RUN / "checkpoint_0532" / "adapter"
weights = adapter_dir / "adapter_model.safetensors"
assert weights.exists() and (adapter_dir / "adapter_config.json").exists(), "Adapter files missing: STOP."
FINAL_T = fitted["Trained Gemma"][0]
config = dict(
    status="PRIMARY: adopted by the pre-registered rule (adoption_decision.json)",
    model_id="google/gemma-4-31B-it", revision="842da3794eaa0b77d5f08bae87a17459d91ff475",
    adapter_dir=str(adapter_dir), adapter_sha256=hashlib.sha256(weights.read_bytes()).hexdigest(),
    final_threshold=float(FINAL_T),
    threshold_rule="single cutoff fitted on all 10 Swedish dev words, predeclared grid, ties -> lower cutoff",
    prompt="v1, unchanged (language-templated system + user message)",
    scoring="bf16 base + fp32 LoRA, bf16 autocast, batch 2, left padding, logits_to_keep=1, margin = logit(Yes) - logit(No)",
    swedish_nested_cv_f1=result["report"]["average"], swedish_original_f1=original["average"],
    training_run=str(FULL_RUN), previous_primary="outputs/frozen/provisional_primary_gemma4_31b.json (kept)")
out = PROJECT / "outputs/frozen/primary_gemma4_31b_lora.json"
assert not out.exists(), "Frozen file already exists: STOP, do not overwrite."
save_json(out, config)
print(f"Final cutoff for trained Gemma : {FINAL_T:.2f}  (held-out folds chose 0.25 or 0.75)")
print(f"Adapter                        : {adapter_dir}")
print(f"Adapter sha256                 : {config['adapter_sha256'][:16]}...")
print(f"Saved                          : {out}")
print("The old 10.5 config is kept untouched as the fallback.")

section("SUMMARY")
print(f"Primary system: Gemma 4 31B + LoRA (step 532), cutoff {FINAL_T:.2f}, Swedish held-out F1 "
      f"{result['report']['average']:.4f} (was {original['average']:.4f}).")
print("Next: Runtime > Disconnect and delete runtime. Then notebook 09 with the adapter.")

In [ ]:
import sys, json, hashlib, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bcubed"], check=True)
from pathlib import Path
from collections import defaultdict

import numpy as np
from scipy.special import expit, logit
from sklearn.linear_model import LogisticRegression
from google.colab import drive

drive.mount("/content/drive")

PROJECT = Path("/content/drive/MyDrive/semeval2027/semeval2027_colab")
for folder in (PROJECT / "src", PROJECT / "notebooks"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))

from project_utils import new_run, save_json
from semeval_task3.data import load_dataset
from semeval_task3.evaluation import (
    score, read_submission, write_submission,
    word_bootstrap, official_score,
)
from semeval_task3.__main__ import check_agreement

FULL_RUN = (
    PROJECT / "outputs/runs/step09_lora_full_pilot"
    / "20261105T232211Z_6b04a51b"
)
PILOT = (
    PROJECT / "outputs/runs/step07_pilot_dataset"
    / "20261105T223146Z_1dd5a63b"
)

SEL_FILE = PILOT / "pilot_selection_ru_dev.jsonl"
SEL_SCORES = FULL_RUN / "checkpoint_0532/selection_scores.json"
SV_SCORES = FULL_RUN / "SV_scores.json"
REFERENCE = FULL_RUN / "lora_selected_SV.zip"

for path in (SEL_FILE, SEL_SCORES, SV_SCORES, REFERENCE):
    assert path.is_file(), f"Missing file: {path}"

OUT = new_run(PROJECT, "step10_word_adaptive_decisions")
EPS = 1e-8
MAX_ITER = 1000
TOL = 1e-8

# ------------------------------------------------------------
# 1. Fit probability calibration on Russian selection ONLY.
# ------------------------------------------------------------
manifest = json.loads((PILOT / "manifest.json").read_text())
assert hashlib.sha256(SEL_FILE.read_bytes()).hexdigest() == (
    manifest["sha256"][SEL_FILE.name]
), "Russian selection file changed."

with SEL_FILE.open(encoding="utf-8") as handle:
    selection = [json.loads(line) for line in handle if line.strip()]

saved = json.loads(SEL_SCORES.read_text())
assert saved["model_tag"] == "selection_step_532"
assert len(selection) == 2946
assert len({str(r["id"]) for r in selection}) == len(selection)
assert set(saved["scores"]) == {str(r["id"]) for r in selection}

x = np.array([
    saved["scores"][str(r["id"])]["margin"] for r in selection
], dtype=float)
y = np.array([int(r["label"]) for r in selection])

assert np.isfinite(x).all()
assert set(y) == {0, 1}

calibrator = LogisticRegression(
    C=1e6, max_iter=1000, solver="lbfgs", random_state=2027
).fit(x.reshape(-1, 1), y)

assert int(calibrator.n_iter_[0]) < 1000, "Calibration did not converge."

A = float(calibrator.coef_[0, 0])
B = float(calibrator.intercept_[0])
SOURCE_PRIOR = float(y.mean())
assert A > 0 and 0 < SOURCE_PRIOR < 1

print("Russian calibration fitted.")
print(f"Source positive rate: {SOURCE_PRIOR:.6f}")
print(f"Probability mapping: sigmoid({A:.6f} × margin + {B:.6f})")

# ------------------------------------------------------------
# 2. Read Swedish inputs WITHOUT gold labels.
# ------------------------------------------------------------
sv_inputs = load_dataset(PROJECT / "work/data/dev", with_gold=False)
sv_saved = json.loads(SV_SCORES.read_text())
assert sv_saved["model_tag"] == "chosen_532_SV"

expected_ids = {"|".join(u.key) for u in sv_inputs.usages}
assert set(sv_saved["scores"]) == expected_ids
assert len(expected_ids) == 1983

groups = defaultdict(list)
for usage in sv_inputs.usages:
    groups[(usage.language, usage.word)].append(usage.key)

def adjust_prior(probabilities):
    """Estimate a group's prior using EM; record convergence."""
    p = np.clip(np.asarray(probabilities, dtype=float), EPS, 1 - EPS)
    evidence = logit(p) - logit(SOURCE_PRIOR)
    prior = SOURCE_PRIOR
    converged = False

    for iteration in range(1, MAX_ITER + 1):
        adjusted = expit(evidence + logit(prior))
        updated = float(np.clip(adjusted.mean(), EPS, 1 - EPS))

        if abs(updated - prior) < TOL:
            prior = updated
            converged = True
            break
        prior = updated

    adjusted = expit(evidence + logit(prior))

    # Fixed fallback policy, chosen before viewing Swedish results.
    used = adjusted if converged else p
    return used, {
        "em_prior": prior,
        "converged": converged,
        "iterations": iteration,
        "near_boundary": prior < 0.001 or prior > 0.999,
        "fallback_used": not converged,
    }

predictions = {}
diagnostics = {}

for (language, word), keys in sorted(groups.items()):
    margins = np.array([
        sv_saved["scores"]["|".join(key)]["margin"] for key in keys
    ], dtype=float)
    assert np.isfinite(margins).all()

    calibrated = expit(A * margins + B)
    probabilities, em = adjust_prior(calibrated)

    order = np.argsort(-probabilities, kind="stable")
    cumulative = np.cumsum(probabilities[order])
    k_values = np.arange(1, len(keys) + 1)

    # Approximate expected F1, not exact expected F1.
    utility = 2 * cumulative / (k_values + probabilities.sum())
    utility = np.concatenate(([0.0], utility))
    best_k = int(np.argmax(utility))

    chosen = set(order[:best_k].tolist())
    for index, key in enumerate(keys):
        predictions[key] = int(index in chosen)

    diagnostics[f"{language}|{word}"] = {
        **em,
        "usages": len(keys),
        "estimated_positives": float(probabilities.sum()),
        "predicted_positives": best_k,
        "approximate_expected_f1": float(utility[best_k]),
    }

# Save predictions BEFORE loading Swedish labels.
archive = OUT / "word_adaptive_SV.zip"
write_submission(archive, predictions)

save_json(OUT / "prediction_manifest.json", {
    "method": "Russian Platt calibration + per-word EM + approximate expected-F1",
    "calibration": {
        "a": A, "b": B, "source_prior": SOURCE_PRIOR,
        "source": "Russian selection, checkpoint 532",
    },
    "em": {
        "max_iterations": MAX_ITER,
        "tolerance": TOL,
        "epsilon": EPS,
        "nonconvergence_policy": "use calibrated probabilities without EM",
    },
    "source_sha256": {
        str(path): hashlib.sha256(path.read_bytes()).hexdigest()
        for path in (SEL_FILE, SEL_SCORES, SV_SCORES, REFERENCE)
    },
    "prediction_sha256": hashlib.sha256(archive.read_bytes()).hexdigest(),
    "swedish_labels_used_to_construct_predictions": False,
    "per_word": diagnostics,
})

# ------------------------------------------------------------
# 3. Evaluate the already-saved predictions.
# ------------------------------------------------------------
sv_gold = load_dataset(PROJECT / "work/data/dev", with_gold=True)
gold = sv_gold.require_gold()

reference = score(gold, read_submission(REFERENCE, sv_gold.keys))
assert abs(reference["average"] - 0.704306) < 1e-5

candidate = score(gold, predictions)
paired = word_bootstrap(candidate, other=reference)

old_by_word = {
    (r["language"], r["word"]): r
    for r in reference["per_word"]
}
wins = 0

print("\nSWEDISH RESULTS")
print(
    f"{'word':<12}{'current':>9}{'candidate':>11}{'change':>10}"
    f"{'est. Yes':>10}{'true Yes':>10}{'pred Yes':>10}"
)

for row in candidate["per_word"]:
    key = (row["language"], row["word"])
    old = old_by_word[key]
    change = row["f1"] - old["f1"]
    wins += change > 1e-9
    info = diagnostics[f"{key[0]}|{key[1]}"]

    print(
        f"{row['word']:<12}{old['f1']:>9.4f}{row['f1']:>11.4f}"
        f"{change:>+10.4f}{info['estimated_positives']:>10.1f}"
        f"{row['positives']:>10}{info['predicted_positives']:>10}"
    )

result = {
    "current_f1": reference["average"],
    "candidate_f1": candidate["average"],
    "paired_difference": paired,
    "words_improved": int(wins),
    "exceeds_0_76": bool(candidate["average"] > 0.76),
    "report": candidate,
    "automatic_adoption": False,
    "status": "experimental candidate; frozen primary system unchanged",
}
save_json(OUT / "results.json", result)

# Verify against the official scorer.
official = official_score(
    archive,
    PROJECT / "work/data/dev",
    PROJECT / "work/official-scorer",
)
check_agreement(candidate, official)
save_json(OUT / "official_score.json", official)

print("\nOfficial scorer agreement: PASS")
print(f"Current F1:   {reference['average']:.6f}")
print(f"Candidate F1: {candidate['average']:.6f}")
print(
    f"Change: {paired['estimate'] * 100:+.2f} F1 points; "
    f"95% paired interval "
    f"[{paired['lower'] * 100:+.2f}, {paired['upper'] * 100:+.2f}]"
)
print(f"Words improved: {wins}/10")
print("Above 0.76:", "YES" if result["exceeds_0_76"] else "NO")
print("EM fallbacks:", sum(v["fallback_used"] for v in diagnostics.values()))
print("EM near-boundary estimates:", sum(
    v["near_boundary"] for v in diagnostics.values()
))
print("\nSaved:", OUT)

In [ ]:
import sys, json, hashlib, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bcubed"], check=True)
from pathlib import Path
import numpy as np
from sklearn.linear_model import LogisticRegression
from google.colab import drive

drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/semeval2027/semeval2027_colab")
for folder in (PROJECT / "src", PROJECT / "notebooks"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))

from project_utils import new_run, save_json, read_scores, frozen, save_evaluation
from semeval_task3.data import load_dataset
from semeval_task3.cv import Candidate, nested_word_cv
from semeval_task3.evaluation import score, read_submission, word_bootstrap

def section(t):
    print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78)

FULL_RUN = PROJECT / "outputs/runs/step09_lora_full_pilot/20261105T232211Z_6b04a51b"
PILOT = PROJECT / "outputs/runs/step07_pilot_dataset/20261105T223146Z_1dd5a63b"
BASELINE = PROJECT / "outputs/runs/step08_gemma_unchanged_baseline/20261105T224131Z_63bba2d9"
ORIG_SV = PROJECT / "outputs/runs/step04_gemma4_31b/20261105T164154Z_48096e67/gemma_raw_scores.json"
WEIGHTS = (0.0, 0.25, 0.5, 0.75, 1.0)          # weight on TRAINED Gemma; 1.0 = current system alone
GRID = tuple([-1000.0] + np.round(np.arange(-60, 60.001, 0.25), 2).tolist() + [1000.0])
OUT = new_run(PROJECT, "step10_original_plus_trained")

# ---------------------------------------------------------------------------
section("1. PUT BOTH MODELS ON THE SAME SCALE (log-odds fitted on RUSSIAN selection only)")
manifest = json.loads((PILOT / "manifest.json").read_text())
SEL = PILOT / "pilot_selection_ru_dev.jsonl"
assert hashlib.sha256(SEL.read_bytes()).hexdigest() == manifest["sha256"][SEL.name], "Selection file changed."
sel = [json.loads(l) for l in SEL.open(encoding="utf-8") if l.strip()]
ids = [str(r["id"]) for r in sel]
y = np.array([int(r["label"]) for r in sel])
orig_sel = json.loads((BASELINE / "selection_scores.json").read_text())
trained_sel = json.loads((FULL_RUN / "checkpoint_0532" / "selection_scores.json").read_text())
assert trained_sel["model_tag"] == "selection_step_532"
assert set(orig_sel) == set(trained_sel["scores"]) == set(ids) and len(ids) == 2946
scales = {}
for name, x in (("original", [orig_sel[i]["yes_minus_no"] for i in ids]),
                ("trained", [trained_sel["scores"][i]["margin"] for i in ids])):
    x = np.asarray(x, dtype=float)
    assert np.isfinite(x).all()
    lr = LogisticRegression(C=1e6, max_iter=1000).fit(x.reshape(-1, 1), y)
    assert int(lr.n_iter_[0]) < 1000, f"{name} calibration did not converge"
    scales[name] = (float(lr.coef_[0, 0]), float(lr.intercept_[0]))
    assert scales[name][0] > 0
    print(f"{name:<9} Gemma: log-odds = {scales[name][0]:.4f} x score {scales[name][1]:+.4f}")
r = np.corrcoef([orig_sel[i]["yes_minus_no"] for i in ids], [trained_sel["scores"][i]["margin"] for i in ids])[0, 1]
print(f"Correlation of the two models' scores on Russian selection: {r:.3f} (lower = more room to complement)")

# ---------------------------------------------------------------------------
section("2. SWEDISH SCORES FOR BOTH MODELS")
sv = load_dataset(PROJECT / "work/data/dev", with_gold=True)
orig = read_scores(ORIG_SV, sv)
tr_saved = json.loads((FULL_RUN / "SV_scores.json").read_text())
assert tr_saved["model_tag"] == "chosen_532_SV"
trained = {u.key: float(tr_saved["scores"]["|".join(u.key)]["margin"]) for u in sv.usages}
assert set(orig) == set(trained) == sv.keys and len(sv.keys) == 1983
lo = {"original": {k: scales["original"][0] * v + scales["original"][1] for k, v in orig.items()},
      "trained": {k: scales["trained"][0] * v + scales["trained"][1] for k, v in trained.items()}}
keys = sorted(sv.keys)
print(f"Swedish usages: {len(keys)} | score correlation between models: "
      f"{np.corrcoef([orig[k] for k in keys], [trained[k] for k in keys])[0, 1]:.3f}")
for name in lo:
    v = np.array(list(lo[name].values()))
    print(f"  {name:<9} log-odds range {v.min():+.1f} to {v.max():+.1f}")
    assert np.abs(v).max() < 60, "Log-odds outside the predeclared cutoff grid: STOP and paste this output."

# ---------------------------------------------------------------------------
section("3. NESTED WORD-LEVEL SELECTION OF WEIGHT AND CUTOFF (labels of the held-out word never used)")
print(f"Weights tried (on trained Gemma): {WEIGHTS} | cutoffs: {len(GRID)} predeclared values")
candidates = []
for w in WEIGHTS:
    combo = {k: w * lo["trained"][k] + (1 - w) * lo["original"][k] for k in keys}
    candidates.append(Candidate(f"w_trained_{w:.2f}", frozen(combo), GRID))
result = nested_word_cv(sv, candidates)
print("\nWhat the inner folds chose for each held-out word:")
for s in result["selections"]:
    print(f"  {s['test_group'][1]:<10} weight/cutoff: {s['candidate']:<16} cutoff {s['threshold']:+.2f}  (inner F1 {s['inner_f1']:.4f})")
save_evaluation(OUT, "original_plus_trained_SV", result, sv, PROJECT, official=True)

# ---------------------------------------------------------------------------
section("4. COMPARE WITH THE CURRENT SYSTEM (trained Gemma alone, 0.7043)")
current = score(sv.require_gold(), read_submission(FULL_RUN / "lora_selected_SV.zip", sv.keys))
assert abs(current["average"] - 0.704306) < 1e-5
new = result["report"]
paired = word_bootstrap(new, other=current)
cw = {x["word"]: x["f1"] for x in current["per_word"]}
nw = {x["word"]: x["f1"] for x in new["per_word"]}
better = 0
print(f"{'word':<11}{'current':>9}{'combined':>10}{'change':>9}")
for w in sorted(cw):
    d = nw[w] - cw[w]
    better += d > 1e-9
    print(f"{w:<11}{cw[w]:>9.4f}{nw[w]:>10.4f}{d:>+9.4f}")
need = current["average"] + 0.02
r1, r2 = new["average"] >= need, better >= 6
print(f"\nSwedish F1: current {current['average']:.4f} -> combined {new['average']:.4f} "
      f"(change {paired['estimate']:+.4f}, 95% range {paired['lower']:+.4f} to {paired['upper']:+.4f})")
print(f"Rule 1  F1 >= {need:.4f}              : {'PASS' if r1 else 'FAIL'}")
print(f"Rule 2  better on >= 6 of 10 words : {better}/10 -> {'PASS' if r2 else 'FAIL'}")
save_json(OUT / "comparison.json", dict(scales=scales, weights=WEIGHTS, current_f1=current["average"],
          combined_f1=new["average"], paired=paired, words_better=int(better), adopt=bool(r1 and r2),
          selections=result["selections"], status="experimental; frozen primary system unchanged"))

section("SUMMARY")
print("ADOPT the combination." if (r1 and r2) else "KEEP trained Gemma alone (0.7043). Combination recorded as a tested idea.")
print("Saved:", OUT)